⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Spark SQL Conversion\n",
        "\n",
        "**Source File:** `SILOS_SIL_INVENTORYPRODUCTDIMENSION.txt`\n",
        "**Conversion Timestamp:** `2024-07-30T12:00:00Z`\n",
        "\n",
        "This notebook contains the converted Spark SQL logic from the original ODI session. It performs incremental updates to the `W_INVENTORY_PRODUCT_D` dimension table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"WH_DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_USAGE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"LOW_DATE\", \"1900-01-01 00:00:00\")\n",
        "dbutils.widgets.text(\"SOURCE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"TARGET_CODE\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"EXECUTION_ID\", \"\")\n",
        "dbutils.widgets.text(\"PRUNE_DAYS\", \"0\")\n",
        "dbutils.widgets.text(\"IS_INCREMENTAL\", \"Y\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_check_load_status AS\n",
        "SELECT\n",
        "    CASE\n",
        "        WHEN COUNT(*) > 0 THEN 'Y'\n",
        "        ELSE 'N'\n",
        "    END AS LOAD_STATUS\n",
        "FROM\n",
        "    workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE\n",
        "    package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "    AND (datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "         OR datasource_num_id = ${WH_DATASOURCE_NUM_ID})\n",
        "    AND etl_usage_code = '${ETL_USAGE_CODE}'\n",
        "    AND committed = '1';"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_low_date AS\n",
        "SELECT to_timestamp('${LOW_DATE}', 'yyyy-MM-dd HH:mm:ss') AS etl_low_date;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "display(spark.sql(\"SELECT * FROM v_check_load_status;\"))\n",
        "display(spark.sql(\"SELECT * FROM v_etl_low_date;\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Category Lookup Update (SCEN_TASK_NO {2})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "USING (\n",
        "    SELECT DISTINCT\n",
        "        X.integration_id,\n",
        "        X.inv_prod_cat1,\n",
        "        Y.inv_prod_cat1_wid\n",
        "    FROM\n",
        "        (\n",
        "            SELECT\n",
        "                B.integration_id,\n",
        "                A.integration_id AS inv_prod_cat1\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp AS A,\n",
        "                workspace.prxbi_dw.w_inventory_product_d AS B\n",
        "            WHERE\n",
        "                    CONCAT_WS('~', A.inventory_item_id, A.organization_id) = B.integration_id\n",
        "                AND A.integration_id <> B.inv_prod_cat1\n",
        "        ) AS X,\n",
        "        (\n",
        "            SELECT\n",
        "                P.integration_id,\n",
        "                Q.row_wid AS inv_prod_cat1_wid\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp AS P,\n",
        "                workspace.prxbi_dw.w_prod_cat_dh AS Q\n",
        "            WHERE\n",
        "                Q.integration_id = P.integration_id\n",
        "        ) AS Y\n",
        "    WHERE\n",
        "        X.inv_prod_cat1 = Y.integration_id\n",
        ") AS S\n",
        "ON (T.integration_id = S.integration_id)\n",
        "WHEN MATCHED THEN UPDATE SET\n",
        "    T.inv_prod_cat1 = S.inv_prod_cat1,\n",
        "    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Metadata and Initialization (SCEN_TASK_NO {10} - {50})\n",
        "\n",
        "Oracle PL/SQL blocks and session settings are not applicable to Databricks Spark SQL and have been removed."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error Table (SCEN_TASK_NO {60} - {70})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {60}: Drop error table (Oracle purge removed, E$ tables are typically persistent)\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.e_inventory_product_d;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {70}: Creates the error table\n",
        "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_inventory_product_d\n",
        "(\n",
        "    ora_err_number        BIGINT,\n",
        "    ora_err_mesg          STRING,\n",
        "    ora_err_rowid         STRING,\n",
        "    ora_err_optyp         STRING,\n",
        "    ora_err_tag           STRING,\n",
        "    ind_update            STRING,\n",
        "    diagnostic_rowid      STRING,\n",
        "    error_type_ind        STRING,\n",
        "    autocorrect_ind       STRING  DEFAULT 'N',\n",
        "    autocorrect_code      STRING,\n",
        "    autocorrect_desc      STRING,\n",
        "    committed             STRING DEFAULT '0',\n",
        "    row_wid               STRING,\n",
        "    product_wid           STRING,\n",
        "    inventory_org_wid     STRING,\n",
        "    plant_loc_wid         STRING,\n",
        "    product_num           STRING,\n",
        "    abc_ind               STRING,\n",
        "    planner_code          STRING,\n",
        "    procurement_type_code STRING,\n",
        "    spc_proc_type_code    STRING,\n",
        "    buyer_code            STRING,\n",
        "    buyer_name            STRING,\n",
        "    commodity_code        STRING,\n",
        "    commodity_uom_code    STRING,\n",
        "    profit_center_num     STRING,\n",
        "    reorder_point         STRING,\n",
        "    safety_stock_level    STRING,\n",
        "    min_lot_size          STRING,\n",
        "    max_lot_size          STRING,\n",
        "    fixed_lot_size        STRING,\n",
        "    max_stock_level       STRING,\n",
        "    lot_ordering_cost     STRING,\n",
        "    mrp_time_fence        STRING,\n",
        "    ext_procure_time      STRING,\n",
        "    internal_mfg_time     STRING,\n",
        "    max_storage_days      STRING,\n",
        "    mrp_profile_code      STRING,\n",
        "    mrp_type_code         STRING,\n",
        "    mrp_grp_code          STRING,\n",
        "    lot_size_code         STRING,\n",
        "    backflush_ind         STRING,\n",
        "    qa_inspect_ind        STRING,\n",
        "    repetitive_mfg_ind    STRING,\n",
        "    bulk_item_ind         STRING,\n",
        "    forecast_period       STRING,\n",
        "    mfg_uom_code          STRING,\n",
        "    issue_uom_code        STRING,\n",
        "    manufacturing_place   STRING,\n",
        "    loading_type_code     STRING,\n",
        "    int_store_loc_code    STRING,\n",
        "    ext_store_loc_code    STRING,\n",
        "    active_flg            STRING,\n",
        "    created_by_wid        STRING,\n",
        "    changed_by_wid        STRING,\n",
        "    created_on_dt         STRING,\n",
        "    changed_on_dt         STRING,\n",
        "    aux1_changed_on_dt    STRING,\n",
        "    aux2_changed_on_dt    STRING,\n",
        "    aux3_changed_on_dt    STRING,\n",
        "    aux4_changed_on_dt    STRING,\n",
        "    src_eff_from_dt       STRING,\n",
        "    src_eff_to_dt         STRING,\n",
        "    effective_from_dt     STRING,\n",
        "    effective_to_dt       STRING,\n",
        "    current_flg           STRING,\n",
        "    w_insert_dt           STRING,\n",
        "    w_update_dt           STRING,\n",
        "    datasource_num_id     STRING,\n",
        "    etl_proc_wid          STRING,\n",
        "    integration_id        STRING,\n",
        "    tenant_id             STRING,\n",
        "    x_custom              STRING,\n",
        "    inv_prod_cat1         STRING,\n",
        "    inv_prod_cat2         STRING,\n",
        "    inv_prod_cat3         STRING,\n",
        "    inv_prod_cat4         STRING,\n",
        "    inv_prod_cat5         STRING,\n",
        "    inv_prod_cat6         STRING,\n",
        "    inv_prod_cat7         STRING,\n",
        "    inv_prod_cat8         STRING,\n",
        "    inv_prod_cat9         STRING,\n",
        "    inv_prod_cat10        STRING,\n",
        "    inv_prod_cat1_wid     STRING,\n",
        "    inv_prod_cat2_wid     STRING,\n",
        "    inv_prod_cat3_wid     STRING,\n",
        "    inv_prod_cat4_wid     STRING,\n",
        "    inv_prod_cat5_wid     STRING,\n",
        "    inv_prod_cat6_wid     STRING,\n",
        "    inv_prod_cat7_wid     STRING,\n",
        "    inv_prod_cat8_wid     STRING,\n",
        "    inv_prod_cat9_wid     STRING,\n",
        "    inv_prod_cat10_wid    STRING,\n",
        "    invoiceable_item_flag STRING,\n",
        "    invoice_enabled_flag  STRING,\n",
        "    primary_uom_code      STRING,\n",
        "    c_primary_uom_code    STRING,\n",
        "    unspsc_code           STRING,\n",
        "    unspsc_inv_prod_cat_wid STRING,\n",
        "    commodity_name        STRING,\n",
        "    commodity_uom_name    STRING,\n",
        "    ext_store_loc_name    STRING,\n",
        "    int_store_loc_name    STRING,\n",
        "    issue_uom_name        STRING,\n",
        "    loading_type_name     STRING,\n",
        "    lot_size_name         STRING,\n",
        "    mfg_uom_name          STRING,\n",
        "    mrp_grp_name          STRING,\n",
        "    mrp_profile_name      STRING,\n",
        "    mrp_type_name         STRING,\n",
        "    planner_name          STRING,\n",
        "    primary_uom_name      STRING,\n",
        "    procurement_type_name STRING,\n",
        "    profit_center_name    STRING,\n",
        "    spc_proc_type_name    STRING,\n",
        "    status_code           STRING,\n",
        "    w_status_code         STRING,\n",
        "    product_type_code     STRING,\n",
        "    make_buy_ind          STRING,\n",
        "    fixed_lead_time       STRING,\n",
        "    variable_lead_time    STRING,\n",
        "    cumulative_total_lead_time STRING,\n",
        "    preprocessing_lead_time    STRING,\n",
        "    postprocessing_lead_time   STRING,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence      STRING,\n",
        "    x_organization_name   STRING,\n",
        "    x_product_desc        STRING,\n",
        "    x_uom_desc            STRING,\n",
        "    x_inv_item_flg        STRING,\n",
        "    x_stock_item_flg      STRING,\n",
        "    x_trans_flg           STRING,\n",
        "    x_rev_flg             STRING,\n",
        "    x_cost_flg            STRING,\n",
        "    x_gcoa_acct           STRING,\n",
        "    x_gcoa_prod           STRING,\n",
        "    x_tax_cat             STRING,\n",
        "    organization_id       STRING,\n",
        "    x_gcoa_loc_acct       STRING,\n",
        "    delete_flg            STRING\n",
        ")\n",
        "USING DELTA;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table (SCEN_TASK_NO {110} - {130})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {110}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {120}: Creates the flow table\n",
        "CREATE TABLE workspace.prxbi_dw.i_inventory_product_d_flow\n",
        "(\n",
        "    src_eff_from_dt       TIMESTAMP,\n",
        "    datasource_num_id     BIGINT,\n",
        "    integration_id        STRING,\n",
        "    row_wid               BIGINT,\n",
        "    product_wid           BIGINT,\n",
        "    inventory_org_wid     BIGINT,\n",
        "    plant_loc_wid         BIGINT,\n",
        "    product_num           STRING,\n",
        "    abc_ind               STRING,\n",
        "    planner_code          STRING,\n",
        "    procurement_type_code STRING,\n",
        "    spc_proc_type_code    STRING,\n",
        "    buyer_code            STRING,\n",
        "    buyer_name            STRING,\n",
        "    commodity_code        STRING,\n",
        "    commodity_uom_code    STRING,\n",
        "    profit_center_num     STRING,\n",
        "    reorder_point         BIGINT,\n",
        "    safety_stock_level    BIGINT,\n",
        "    min_lot_size          BIGINT,\n",
        "    max_lot_size          BIGINT,\n",
        "    fixed_lot_size        BIGINT,\n",
        "    max_stock_level       BIGINT,\n",
        "    lot_ordering_cost     BIGINT,\n",
        "    mrp_time_fence        BIGINT,\n",
        "    ext_procure_time      BIGINT,\n",
        "    internal_mfg_time     BIGINT,\n",
        "    max_storage_days      BIGINT,\n",
        "    mrp_profile_code      STRING,\n",
        "    mrp_type_code         STRING,\n",
        "    mrp_grp_code          STRING,\n",
        "    lot_size_code         STRING,\n",
        "    backflush_ind         STRING,\n",
        "    qa_inspect_ind        STRING,\n",
        "    repetitive_mfg_ind    STRING,\n",
        "    bulk_item_ind         STRING,\n",
        "    forecast_period       STRING,\n",
        "    mfg_uom_code          STRING,\n",
        "    issue_uom_code        STRING,\n",
        "    manufacturing_place   STRING,\n",
        "    loading_type_code     STRING,\n",
        "    int_store_loc_code    STRING,\n",
        "    ext_store_loc_code    STRING,\n",
        "    active_flg            STRING,\n",
        "    created_by_wid        BIGINT,\n",
        "    changed_by_wid        BIGINT,\n",
        "    created_on_dt         TIMESTAMP,\n",
        "    changed_on_dt         TIMESTAMP,\n",
        "    aux1_changed_on_dt    TIMESTAMP,\n",
        "    aux2_changed_on_dt    TIMESTAMP,\n",
        "    aux3_changed_on_dt    TIMESTAMP,\n",
        "    aux4_changed_on_dt    TIMESTAMP,\n",
        "    src_eff_to_dt         TIMESTAMP,\n",
        "    effective_from_dt     TIMESTAMP,\n",
        "    effective_to_dt       TIMESTAMP,\n",
        "    delete_flg            STRING,\n",
        "    current_flg           STRING,\n",
        "    w_insert_dt           TIMESTAMP,\n",
        "    w_update_dt           TIMESTAMP,\n",
        "    etl_proc_wid          BIGINT,\n",
        "    tenant_id             STRING,\n",
        "    x_custom              STRING,\n",
        "    inv_prod_cat1         STRING,\n",
        "    inv_prod_cat2         STRING,\n",
        "    inv_prod_cat3         STRING,\n",
        "    inv_prod_cat4         STRING,\n",
        "    inv_prod_cat5         STRING,\n",
        "    inv_prod_cat6         STRING,\n",
        "    inv_prod_cat7         STRING,\n",
        "    inv_prod_cat8         STRING,\n",
        "    inv_prod_cat9         STRING,\n",
        "    inv_prod_cat10        STRING,\n",
        "    inv_prod_cat1_wid     BIGINT,\n",
        "    inv_prod_cat2_wid     BIGINT,\n",
        "    inv_prod_cat3_wid     BIGINT,\n",
        "    inv_prod_cat4_wid     BIGINT,\n",
        "    inv_prod_cat5_wid     BIGINT,\n",
        "    inv_prod_cat6_wid     BIGINT,\n",
        "    inv_prod_cat7_wid     BIGINT,\n",
        "    inv_prod_cat8_wid     BIGINT,\n",
        "    inv_prod_cat9_wid     BIGINT,\n",
        "    inv_prod_cat10_wid    BIGINT,\n",
        "    invoiceable_item_flag STRING,\n",
        "    invoice_enabled_flag  STRING,\n",
        "    primary_uom_code      STRING,\n",
        "    c_primary_uom_code    STRING,\n",
        "    unspsc_code           STRING,\n",
        "    unspsc_inv_prod_cat_wid BIGINT,\n",
        "    commodity_name        STRING,\n",
        "    commodity_uom_name    STRING,\n",
        "    ext_store_loc_name    STRING,\n",
        "    int_store_loc_name    STRING,\n",
        "    issue_uom_name        STRING,\n",
        "    loading_type_name     STRING,\n",
        "    lot_size_name         STRING,\n",
        "    mfg_uom_name          STRING,\n",
        "    mrp_grp_name          STRING,\n",
        "    mrp_profile_name      STRING,\n",
        "    mrp_type_name         STRING,\n",
        "    planner_name          STRING,\n",
        "    primary_uom_name      STRING,\n",
        "    procurement_type_name STRING,\n",
        "    profit_center_name    STRING,\n",
        "    spc_proc_type_name    STRING,\n",
        "    status_code           STRING,\n",
        "    w_status_code         STRING,\n",
        "    product_type_code     STRING,\n",
        "    make_buy_ind          STRING,\n",
        "    fixed_lead_time       BIGINT,\n",
        "    variable_lead_time    BIGINT,\n",
        "    cumulative_total_lead_time BIGINT,\n",
        "    postprocessing_lead_time BIGINT,\n",
        "    preprocessing_lead_time BIGINT,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence      STRING,\n",
        "    x_organization_name   STRING,\n",
        "    x_product_desc        STRING,\n",
        "    x_uom_desc            STRING,\n",
        "    x_inv_item_flg        STRING,\n",
        "    x_stock_item_flg      STRING,\n",
        "    x_trans_flg           STRING,\n",
        "    x_rev_flg             STRING,\n",
        "    x_cost_flg            STRING,\n",
        "    x_gcoa_acct           STRING,\n",
        "    x_gcoa_prod           STRING,\n",
        "    x_tax_cat             STRING,\n",
        "    organization_id       STRING,\n",
        "    x_gcoa_loc_acct       STRING,\n",
        "    ind_update            STRING\n",
        ")\n",
        "USING DELTA;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {130}: Insert into flow table\n",
        "INSERT INTO workspace.prxbi_dw.i_inventory_product_d_flow\n",
        "(\n",
        "    product_wid,\n",
        "    inventory_org_wid,\n",
        "    plant_loc_wid,\n",
        "    product_num,\n",
        "    abc_ind,\n",
        "    planner_code,\n",
        "    procurement_type_code,\n",
        "    spc_proc_type_code,\n",
        "    buyer_code,\n",
        "    buyer_name,\n",
        "    commodity_code,\n",
        "    commodity_uom_code,\n",
        "    profit_center_num,\n",
        "    reorder_point,\n",
        "    safety_stock_level,\n",
        "    min_lot_size,\n",
        "    max_lot_size,\n",
        "    fixed_lot_size,\n",
        "    max_stock_level,\n",
        "    lot_ordering_cost,\n",
        "    mrp_time_fence,\n",
        "    ext_procure_time,\n",
        "    internal_mfg_time,\n",
        "    max_storage_days,\n",
        "    mrp_profile_code,\n",
        "    mrp_type_code,\n",
        "    mrp_grp_code,\n",
        "    lot_size_code,\n",
        "    backflush_ind,\n",
        "    qa_inspect_ind,\n",
        "    repetitive_mfg_ind,\n",
        "    bulk_item_ind,\n",
        "    forecast_period,\n",
        "    mfg_uom_code,\n",
        "    issue_uom_code,\n",
        "    manufacturing_place,\n",
        "    loading_type_code,\n",
        "    int_store_loc_code,\n",
        "    ext_store_loc_code,\n",
        "    active_flg,\n",
        "    created_by_wid,\n",
        "    changed_by_wid,\n",
        "    created_on_dt,\n",
        "    changed_on_dt,\n",
        "    aux1_changed_on_dt,\n",
        "    aux2_changed_on_dt,\n",
        "    aux3_changed_on_dt,\n",
        "    aux4_changed_on_dt,\n",
        "    src_eff_from_dt,\n",
        "    src_eff_to_dt,\n",
        "    effective_from_dt,\n",
        "    delete_flg,\n",
        "    datasource_num_id,\n",
        "    integration_id,\n",
        "    tenant_id,\n",
        "    x_custom,\n",
        "    inv_prod_cat1,\n",
        "    inv_prod_cat2,\n",
        "    inv_prod_cat3,\n",
        "    inv_prod_cat4,\n",
        "    inv_prod_cat5,\n",
        "    inv_prod_cat6,\n",
        "    inv_prod_cat7,\n",
        "    inv_prod_cat8,\n",
        "    inv_prod_cat9,\n",
        "    inv_prod_cat10,\n",
        "    inv_prod_cat1_wid,\n",
        "    inv_prod_cat2_wid,\n",
        "    inv_prod_cat3_wid,\n",
        "    inv_prod_cat4_wid,\n",
        "    inv_prod_cat5_wid,\n",
        "    inv_prod_cat6_wid,\n",
        "    inv_prod_cat7_wid,\n",
        "    inv_prod_cat8_wid,\n",
        "    inv_prod_cat9_wid,\n",
        "    inv_prod_cat10_wid,\n",
        "    invoiceable_item_flag,\n",
        "    invoice_enabled_flag,\n",
        "    primary_uom_code,\n",
        "    c_primary_uom_code,\n",
        "    unspsc_code,\n",
        "    unspsc_inv_prod_cat_wid,\n",
        "    commodity_name,\n",
        "    commodity_uom_name,\n",
        "    ext_store_loc_name,\n",
        "    int_store_loc_name,\n",
        "    issue_uom_name,\n",
        "    loading_type_name,\n",
        "    lot_size_name,\n",
        "    mfg_uom_name,\n",
        "    mrp_grp_name,\n",
        "    mrp_profile_name,\n",
        "    mrp_type_name,\n",
        "    planner_name,\n",
        "    primary_uom_name,\n",
        "    procurement_type_name,\n",
        "    profit_center_name,\n",
        "    spc_proc_type_name,\n",
        "    status_code,\n",
        "    w_status_code,\n",
        "    product_type_code,\n",
        "    make_buy_ind,\n",
        "    fixed_lead_time,\n",
        "    variable_lead_time,\n",
        "    cumulative_total_lead_time,\n",
        "    postprocessing_lead_time,\n",
        "    preprocessing_lead_time,\n",
        "    process_quality_enabled_flg,\n",
        "    x_price_sequence,\n",
        "    x_organization_name,\n",
        "    x_product_desc,\n",
        "    x_uom_desc,\n",
        "    x_inv_item_flg,\n",
        "    x_stock_item_flg,\n",
        "    x_trans_flg,\n",
        "    x_rev_flg,\n",
        "    x_cost_flg,\n",
        "    x_gcoa_acct,\n",
        "    x_gcoa_prod,\n",
        "    x_tax_cat,\n",
        "    organization_id,\n",
        "    x_gcoa_loc_acct,\n",
        "    current_flg,\n",
        "    effective_to_dt,\n",
        "    ind_update\n",
        ")\n",
        "SELECT\n",
        "    C.product_wid,\n",
        "    C.inventory_org_wid,\n",
        "    C.plant_loc_wid,\n",
        "    C.product_num,\n",
        "    C.abc_ind,\n",
        "    C.planner_code,\n",
        "    C.procurement_type_code,\n",
        "    C.spc_proc_type_code,\n",
        "    C.buyer_code,\n",
        "    C.buyer_name,\n",
        "    C.commodity_code,\n",
        "    C.commodity_uom_code,\n",
        "    C.profit_center_num,\n",
        "    C.reorder_point,\n",
        "    C.safety_stock_level,\n",
        "    C.min_lot_size,\n",
        "    C.max_lot_size,\n",
        "    C.fixed_lot_size,\n",
        "    C.max_stock_level,\n",
        "    C.lot_ordering_cost,\n",
        "    C.mrp_time_fence,\n",
        "    C.ext_procure_time,\n",
        "    C.internal_mfg_time,\n",
        "    C.max_storage_days,\n",
        "    C.mrp_profile_code,\n",
        "    C.mrp_type_code,\n",
        "    C.mrp_grp_code,\n",
        "    C.lot_size_code,\n",
        "    C.backflush_ind,\n",
        "    C.qa_inspect_ind,\n",
        "    C.repetitive_mfg_ind,\n",
        "    C.bulk_item_ind,\n",
        "    C.forecast_period,\n",
        "    C.mfg_uom_code,\n",
        "    C.issue_uom_code,\n",
        "    C.manufacturing_place,\n",
        "    C.loading_type_code,\n",
        "    C.int_store_loc_code,\n",
        "    C.ext_store_loc_code,\n",
        "    C.active_flg,\n",
        "    C.created_by_wid,\n",
        "    C.changed_by_wid,\n",
        "    C.created_on_dt,\n",
        "    C.changed_on_dt,\n",
        "    C.aux1_changed_on_dt,\n",
        "    C.aux2_changed_on_dt,\n",
        "    C.aux3_changed_on_dt,\n",
        "    C.aux4_changed_on_dt,\n",
        "    C.src_eff_from_dt,\n",
        "    C.src_eff_to_dt,\n",
        "    C.effective_from_dt,\n",
        "    C.delete_flg,\n",
        "    C.datasource_num_id,\n",
        "    C.integration_id,\n",
        "    C.tenant_id,\n",
        "    C.x_custom,\n",
        "    C.inv_prod_cat1,\n",
        "    C.inv_prod_cat2,\n",
        "    C.inv_prod_cat3,\n",
        "    C.inv_prod_cat4,\n",
        "    C.inv_prod_cat5,\n",
        "    C.inv_prod_cat6,\n",
        "    C.inv_prod_cat7,\n",
        "    C.inv_prod_cat8,\n",
        "    C.inv_prod_cat9,\n",
        "    C.inv_prod_cat10,\n",
        "    C.inv_prod_cat1_wid,\n",
        "    C.inv_prod_cat2_wid,\n",
        "    C.inv_prod_cat3_wid,\n",
        "    C.inv_prod_cat4_wid,\n",
        "    C.inv_prod_cat5_wid,\n",
        "    C.inv_prod_cat6_wid,\n",
        "    C.inv_prod_cat7_wid,\n",
        "    C.inv_prod_cat8_wid,\n",
        "    C.inv_prod_cat9_wid,\n",
        "    C.inv_prod_cat10_wid,\n",
        "    C.invoiceable_item_flag,\n",
        "    C.invoice_enabled_flag,\n",
        "    C.primary_uom_code,\n",
        "    C.c_primary_uom_code,\n",
        "    C.unspsc_code,\n",
        "    C.unspsc_inv_prod_cat_wid,\n",
        "    C.commodity_name,\n",
        "    C.commodity_uom_name,\n",
        "    C.ext_store_loc_name,\n",
        "    C.int_store_loc_name,\n",
        "    C.issue_uom_name,\n",
        "    C.loading_type_name,\n",
        "    C.lot_size_name,\n",
        "    C.mfg_uom_name,\n",
        "    C.mrp_grp_name,\n",
        "    C.mrp_profile_name,\n",
        "    C.mrp_type_name,\n",
        "    C.planner_name,\n",
        "    C.primary_uom_name,\n",
        "    C.procurement_type_name,\n",
        "    C.profit_center_name,\n",
        "    C.spc_proc_type_name,\n",
        "    C.status_code,\n",
        "    C.w_status_code,\n",
        "    C.product_type_code,\n",
        "    C.make_buy_ind,\n",
        "    C.fixed_lead_time,\n",
        "    C.variable_lead_time,\n",
        "    C.cumulative_total_lead_time,\n",
        "    C.postprocessing_lead_time,\n",
        "    C.preprocessing_lead_time,\n",
        "    C.process_quality_enabled_flg,\n",
        "    C.x_price_sequence,\n",
        "    C.x_organization_name,\n",
        "    C.x_product_desc,\n",
        "    C.x_uom_desc,\n",
        "    C.x_inv_item_flg,\n",
        "    C.x_stock_item_flg,\n",
        "    C.x_trans_flg,\n",
        "    C.x_rev_flg,\n",
        "    C.x_cost_flg,\n",
        "    C.x_gcoa_acct,\n",
        "    C.x_gcoa_prod,\n",
        "    C.x_tax_cat,\n",
        "    C.organization_id,\n",
        "    C.x_gcoa_loc_acct,\n",
        "    'Y' AS current_flg,\n",
        "    to_timestamp('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss') AS effective_to_dt,\n",
        "    CASE\n",
        "        WHEN T.integration_id IS NOT NULL\n",
        "             AND (\n",
        "                 T.changed_on_dt = C.changed_on_dt\n",
        "                 OR (T.changed_on_dt IS NULL AND C.changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux1_changed_on_dt = C.aux1_changed_on_dt\n",
        "                 OR (T.aux1_changed_on_dt IS NULL AND C.aux1_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux2_changed_on_dt = C.aux2_changed_on_dt\n",
        "                 OR (T.aux2_changed_on_dt IS NULL AND C.aux2_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux3_changed_on_dt = C.aux3_changed_on_dt\n",
        "                 OR (T.aux3_changed_on_dt IS NULL AND C.aux3_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux4_changed_on_dt = C.aux4_changed_on_dt\n",
        "                 OR (T.aux4_changed_on_dt IS NULL AND C.aux4_changed_on_dt IS NULL)\n",
        "             )\n",
        "        THEN 'N'\n",
        "        WHEN T.integration_id IS NOT NULL THEN 'U'\n",
        "        ELSE 'I'\n",
        "    END AS ind_update\n",
        "FROM\n",
        "    (\n",
        "        SELECT\n",
        "            COALESCE(inline_view.scd1_wid_1, 0) AS product_wid,\n",
        "            COALESCE(inline_view.scd1_wid, 0) AS inventory_org_wid,\n",
        "            COALESCE(inline_view.row_wid, 0) AS plant_loc_wid,\n",
        "            inline_view.product_num AS product_num,\n",
        "            inline_view.abc_ind AS abc_ind,\n",
        "            COALESCE(inline_view.planner_code, '__NOT_APPLICABLE__') AS planner_code,\n",
        "            COALESCE(inline_view.procurement_type_code, '__NOT_APPLICABLE__') AS procurement_type_code,\n",
        "            COALESCE(inline_view.spc_proc_type_code, '__NOT_APPLICABLE__') AS spc_proc_type_code,\n",
        "            COALESCE(inline_view.buyer_code, '__NOT_APPLICABLE__') AS buyer_code,\n",
        "            inline_view.buyer_name AS buyer_name,\n",
        "            COALESCE(inline_view.commodity_code, '__NOT_APPLICABLE__') AS commodity_code,\n",
        "            COALESCE(inline_view.commodity_uom_code, '__NOT_APPLICABLE__') AS commodity_uom_code,\n",
        "            inline_view.profit_center_num AS profit_center_num,\n",
        "            inline_view.reorder_point AS reorder_point,\n",
        "            inline_view.safety_stock_level AS safety_stock_level,\n",
        "            inline_view.min_lot_size AS min_lot_size,\n",
        "            inline_view.max_lot_size AS max_lot_size,\n",
        "            inline_view.fixed_lot_size AS fixed_lot_size,\n",
        "            inline_view.max_stock_level AS max_stock_level,\n",
        "            inline_view.lot_ordering_cost AS lot_ordering_cost,\n",
        "            inline_view.mrp_time_fence AS mrp_time_fence,\n",
        "            inline_view.ext_procure_time AS ext_procure_time,\n",
        "            inline_view.internal_mfg_time AS internal_mfg_time,\n",
        "            inline_view.max_storage_days AS max_storage_days,\n",
        "            COALESCE(inline_view.mrp_profile_code, '__NOT_APPLICABLE__') AS mrp_profile_code,\n",
        "            COALESCE(inline_view.mrp_type_code, '__NOT_APPLICABLE__') AS mrp_type_code,\n",
        "            COALESCE(inline_view.mrp_grp_code, '__NOT_APPLICABLE__') AS mrp_grp_code,\n",
        "            COALESCE(inline_view.lot_size_code, '__NOT_APPLICABLE__') AS lot_size_code,\n",
        "            inline_view.backflush_ind AS backflush_ind,\n",
        "            inline_view.qa_inspect_ind AS qa_inspect_ind,\n",
        "            inline_view.repetitive_mfg_ind AS repetitive_mfg_ind,\n",
        "            inline_view.bulk_item_ind AS bulk_item_ind,\n",
        "            inline_view.forecast_period AS forecast_period,\n",
        "            COALESCE(inline_view.mfg_uom_code, '__NOT_APPLICABLE__') AS mfg_uom_code,\n",
        "            COALESCE(inline_view.issue_uom_code, '__NOT_APPLICABLE__') AS issue_uom_code,\n",
        "            inline_view.manufacturing_place AS manufacturing_place,\n",
        "            COALESCE(inline_view.loading_type_code, '__NOT_APPLICABLE__') AS loading_type_code,\n",
        "            COALESCE(inline_view.int_store_loc_code, '__NOT_APPLICABLE__') AS int_store_loc_code,\n",
        "            COALESCE(inline_view.ext_store_loc_code, '__NOT_APPLICABLE__') AS ext_store_loc_code,\n",
        "            inline_view.active_flg AS active_flg,\n",
        "            COALESCE(lkp_w_user_d_lkp_w_user_d_cr_1.row_wid, 0) AS created_by_wid,\n",
        "            COALESCE(inline_view.row_wid_1, 0) AS changed_by_wid,\n",
        "            inline_view.created_on_dt AS created_on_dt,\n",
        "            inline_view.changed_on_dt AS changed_on_dt,\n",
        "            inline_view.aux1_changed_on_dt AS aux1_changed_on_dt,\n",
        "            inline_view.aux2_changed_on_dt AS aux2_changed_on_dt,\n",
        "            inline_view.aux3_changed_on_dt AS aux3_changed_on_dt,\n",
        "            inline_view.aux4_changed_on_dt AS aux4_changed_on_dt,\n",
        "            inline_view.src_eff_from_dt AS src_eff_from_dt,\n",
        "            inline_view.src_eff_to_dt AS src_eff_to_dt,\n",
        "            COALESCE(inline_view.src_eff_from_dt, to_timestamp('${LOW_DATE}', 'yyyy-MM-dd HH:mm:ss')) AS effective_from_dt,\n",
        "            (CASE WHEN inline_view.delete_flg = 'Y' THEN 'Y' ELSE 'N' END) AS delete_flg,\n",
        "            inline_view.datasource_num_id AS datasource_num_id,\n",
        "            inline_view.integration_id AS integration_id,\n",
        "            inline_view.tenant_id AS tenant_id,\n",
        "            inline_view.x_custom AS x_custom,\n",
        "            inline_view.inv_prod_cat1 AS inv_prod_cat1,\n",
        "            inline_view.inv_prod_cat2 AS inv_prod_cat2,\n",
        "            inline_view.inv_prod_cat3 AS inv_prod_cat3,\n",
        "            inline_view.inv_prod_cat4 AS inv_prod_cat4,\n",
        "            inline_view.inv_prod_cat5 AS inv_prod_cat5,\n",
        "            inline_view.inv_prod_cat6 AS inv_prod_cat6,\n",
        "            inline_view.inv_prod_cat7 AS inv_prod_cat7,\n",
        "            inline_view.inv_prod_cat8 AS inv_prod_cat8,\n",
        "            inline_view.inv_prod_cat9 AS inv_prod_cat9,\n",
        "            inline_view.inv_prod_cat10 AS inv_prod_cat10,\n",
        "            (CASE WHEN inline_view.inv_prod_cat1 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat1_row_wid, 0) END) AS inv_prod_cat1_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat2 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat2_row_wid, 0) END) AS inv_prod_cat2_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat3 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat3_row_wid, 0) END) AS inv_prod_cat3_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat4 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat4_row_wid, 0) END) AS inv_prod_cat4_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat5 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat5_row_wid, 0) END) AS inv_prod_cat5_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat6 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat6_row_wid, 0) END) AS inv_prod_cat6_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat7 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat7_row_wid, 0) END) AS inv_prod_cat7_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat8 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat8_row_wid, 0) END) AS inv_prod_cat8_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat9 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat9_row_wid, 0) END) AS inv_prod_cat9_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat10 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat10_row_wid, 0) END) AS inv_prod_cat10_wid,\n",
        "            COALESCE(inline_view.invoiceable_item_flag, 'N') AS invoiceable_item_flag,\n",
        "            COALESCE(inline_view.invoice_enabled_flag, 'N') AS invoice_enabled_flag,\n",
        "            COALESCE(inline_view.primary_uom_code, '__NOT_APPLICABLE__') AS primary_uom_code,\n",
        "            COALESCE(\n",
        "                (SELECT T1.trg_domain_member_code FROM workspace.prxbi_dw.w_domain_member_map_g AS T1\n",
        "                 WHERE T1.src_domain_code = '${SOURCE_CODE}'\n",
        "                   AND T1.src_domain_member_code = COALESCE(inline_view.primary_uom_code, '__UNASSIGNED__')\n",
        "                   AND T1.src_datasource_num_id IN (inline_view.datasource_num_id, 999)\n",
        "                   AND T1.trg_domain_code = '${TARGET_CODE}'),\n",
        "                (SELECT T2.trg_domain_member_code FROM workspace.prxbi_dw.w_domain_member_map_g AS T2\n",
        "                 WHERE T2.src_domain_code = '${SOURCE_CODE}'\n",
        "                   AND T2.src_domain_member_code = '__ANY__'\n",
        "                   AND T2.src_datasource_num_id IN (inline_view.datasource_num_id, 999)\n",
        "                   AND T2.trg_domain_code = '${TARGET_CODE}'),\n",
        "                (CASE\n",
        "                    WHEN inline_view.primary_uom_code IS NULL THEN COALESCE(\n",
        "                        (SELECT T3.domain_member_code FROM workspace.prxbi_dw.w_domain_member_g AS T3\n",
        "                         WHERE T3.domain_member_code = '__UNASSIGNED__'\n",
        "                           AND T3.domain_code = '${TARGET_CODE}'),\n",
        "                        '__ERROR__'\n",
        "                    )\n",
        "                    ELSE (CASE WHEN '${TARGET_CODE}' = 'W_LANGUAGE' THEN '_ERR' ELSE '__ERROR__' END)\n",
        "                END)\n",
        "            ) AS c_primary_uom_code,\n",
        "            COALESCE(inline_view.unspsc_code, '__NOT_APPLICABLE__') AS unspsc_code,\n",
        "            COALESCE(inline_view.inv_prod_cat_unspsc_row_wid, 0) AS unspsc_inv_prod_cat_wid,\n",
        "            inline_view.commodity_name AS commodity_name,\n",
        "            inline_view.commodity_uom_name AS commodity_uom_name,\n",
        "            inline_view.ext_store_loc_name AS ext_store_loc_name,\n",
        "            inline_view.int_store_loc_name AS int_store_loc_name,\n",
        "            inline_view.issue_uom_name AS issue_uom_name,\n",
        "            inline_view.loading_type_name AS loading_type_name,\n",
        "            inline_view.lot_size_name AS lot_size_name,\n",
        "            inline_view.mfg_uom_name AS mfg_uom_name,\n",
        "            inline_view.mrp_grp_name AS mrp_grp_name,\n",
        "            inline_view.mrp_profile_name AS mrp_profile_name,\n",
        "            inline_view.mrp_type_name AS mrp_type_name,\n",
        "            inline_view.planner_name AS planner_name,\n",
        "            inline_view.primary_uom_name AS primary_uom_name,\n",
        "            inline_view.procurement_type_name AS procurement_type_name,\n",
        "            inline_view.profit_center_name AS profit_center_name,\n",
        "            inline_view.spc_proc_type_name AS spc_proc_type_name,\n",
        "            inline_view.status_code AS status_code,\n",
        "            inline_view.w_status_code AS w_status_code,\n",
        "            inline_view.product_type_code AS product_type_code,\n",
        "            inline_view.make_buy_ind AS make_buy_ind,\n",
        "            inline_view.fixed_lead_time AS fixed_lead_time,\n",
        "            inline_view.variable_lead_time AS variable_lead_time,\n",
        "            inline_view.cumulative_total_lead_time AS cumulative_total_lead_time,\n",
        "            inline_view.preprocessing_lead_time AS postprocessing_lead_time,\n",
        "            inline_view.preprocessing_lead_time AS preprocessing_lead_time,\n",
        "            inline_view.process_quality_enabled_flg AS process_quality_enabled_flg,\n",
        "            inline_view.x_price_sequence AS x_price_sequence,\n",
        "            inline_view.x_organization_name AS x_organization_name,\n",
        "            inline_view.x_product_desc AS x_product_desc,\n",
        "            inline_view.x_uom_desc AS x_uom_desc,\n",
        "            inline_view.x_inv_item_flg AS x_inv_item_flg,\n",
        "            inline_view.x_stock_item_flg AS x_stock_item_flg,\n",
        "            inline_view.x_trans_flg AS x_trans_flg,\n",
        "            inline_view.x_rev_flg AS x_rev_flg,\n",
        "            inline_view.x_cost_flg AS x_cost_flg,\n",
        "            inline_view.x_gcoa_acct AS x_gcoa_acct,\n",
        "            inline_view.x_gcoa_prod AS x_gcoa_prod,\n",
        "            inline_view.x_tax_cat AS x_tax_cat,\n",
        "            inline_view.inventory_org_id AS organization_id,\n",
        "            inline_view.x_gcoa_loc_acct AS x_gcoa_loc_acct\n",
        "        FROM\n",
        "            (\n",
        "                SELECT\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_type_name AS mrp_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_custom AS x_custom,\n",
        "                    sq_w_inventory_product_ds_sq_w.src_eff_to_dt AS src_eff_to_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_procure_time AS ext_procure_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_uom_code AS commodity_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat10 AS inv_prod_cat10,\n",
        "                    sq_w_inventory_product_ds_sq_w.invoice_enabled_flag AS invoice_enabled_flag,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_profile_code AS mrp_profile_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_store_loc_name AS ext_store_loc_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux3_changed_on_dt AS aux3_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.min_lot_size AS min_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.reorder_point AS reorder_point,\n",
        "                    sq_w_inventory_product_ds_sq_w.bulk_item_ind AS bulk_item_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.spc_proc_type_name AS spc_proc_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_code AS commodity_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.repetitive_mfg_ind AS repetitive_mfg_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.loading_type_code AS loading_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.profit_center_name AS profit_center_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.created_by_id AS created_by_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.buyer_code AS buyer_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.int_store_loc_name AS int_store_loc_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_name AS commodity_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.planner_name AS planner_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_storage_days AS max_storage_days,\n",
        "                    sq_w_inventory_product_ds_sq_w.planner_code AS planner_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_lot_size AS max_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_grp_code AS mrp_grp_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_ordering_cost AS lot_ordering_cost,\n",
        "                    sq_w_inventory_product_ds_sq_w.internal_mfg_time AS internal_mfg_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.manufacturing_place AS manufacturing_place,\n",
        "                    sq_w_inventory_product_ds_sq_w.procurement_type_name AS procurement_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.plant_loc_id AS plant_loc_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.changed_by_id AS changed_by_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.backflush_ind AS backflush_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux2_changed_on_dt AS aux2_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.datasource_num_id AS datasource_num_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.spc_proc_type_code AS spc_proc_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_size_code AS lot_size_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.changed_on_dt AS changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_id AS product_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.primary_uom_code AS primary_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_num AS product_num,\n",
        "                    sq_w_inventory_product_ds_sq_w.forecast_period AS forecast_period,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux1_changed_on_dt AS aux1_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.buyer_name AS buyer_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.fixed_lot_size AS fixed_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_grp_name AS mrp_grp_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.procurement_type_code AS procurement_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.primary_uom_name AS primary_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_stock_level AS max_stock_level,\n",
        "                    sq_w_inventory_product_ds_sq_w.safety_stock_level AS safety_stock_level,\n",
        "                    sq_w_inventory_product_ds_sq_w.int_store_loc_code AS int_store_loc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_store_loc_code AS ext_store_loc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.issue_uom_name AS issue_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.abc_ind AS abc_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.profit_center_num AS profit_center_num,\n",
        "                    sq_w_inventory_product_ds_sq_w.created_on_dt AS created_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.unspsc_code AS unspsc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat1 AS inv_prod_cat1,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat3 AS inv_prod_cat3,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat2 AS inv_prod_cat2,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_uom_name AS commodity_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat5 AS inv_prod_cat5,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat4 AS inv_prod_cat4,\n",
        "                    sq_w_inventory_product_ds_sq_w.mfg_uom_name AS mfg_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat7 AS inv_prod_cat7,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat6 AS inv_prod_cat6,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat9 AS inv_prod_cat9,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat8 AS inv_prod_cat8,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux4_changed_on_dt AS aux4_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_size_name AS lot_size_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.tenant_id AS tenant_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.invoiceable_item_flag AS invoiceable_item_flag,\n",
        "                    sq_w_inventory_product_ds_sq_w.inventory_org_id AS inventory_org_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.src_eff_from_dt AS src_eff_from_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.integration_id AS integration_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.issue_uom_code AS issue_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.delete_flg AS delete_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_type_code AS mrp_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.active_flg AS active_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_time_fence AS mrp_time_fence,\n",
        "                    sq_w_inventory_product_ds_sq_w.qa_inspect_ind AS qa_inspect_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.mfg_uom_code AS mfg_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.loading_type_name AS loading_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_profile_name AS mrp_profile_name,\n",
        "                    w_prod_cat_dh1_sq_w_inventory_.row_wid AS inv_prod_cat1_row_wid,\n",
        "                    w_prod_cat_dh2_sq_w_inventory_.row_wid AS inv_prod_cat2_row_wid,\n",
        "                    w_prod_cat_dh3_sq_w_inventory_.row_wid AS inv_prod_cat3_row_wid,\n",
        "                    w_prod_cat_dh4_sq_w_inventory_.row_wid AS inv_prod_cat4_row_wid,\n",
        "                    w_prod_cat_dh5_sq_w_inventory_.row_wid AS inv_prod_cat5_row_wid,\n",
        "                    w_prod_cat_dh6_sq_w_inventory_.row_wid AS inv_prod_cat6_row_wid,\n",
        "                    w_prod_cat_dh7_sq_w_inventory_.row_wid AS inv_prod_cat7_row_wid,\n",
        "                    w_prod_cat_dh8_sq_w_inventory_.row_wid AS inv_prod_cat8_row_wid,\n",
        "                    w_prod_cat_dh_unspsc_sq_w_inve.row_wid AS inv_prod_cat_unspsc_row_wid,\n",
        "                    w_prod_cat_dh9_sq_w_inventory_.row_wid AS inv_prod_cat9_row_wid,\n",
        "                    w_prod_cat_dh10_sq_w_inventory.row_wid AS inv_prod_cat10_row_wid,\n",
        "                    sq_w_inventory_product_ds_sq_w.cumulative_total_lead_time AS cumulative_total_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.fixed_lead_time AS fixed_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.w_status_code AS w_status_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.preprocessing_lead_time AS preprocessing_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.status_code AS status_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.variable_lead_time AS variable_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.make_buy_ind AS make_buy_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.process_quality_enabled_flg AS process_quality_enabled_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_type_code AS product_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_price_sequence AS x_price_sequence,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_organization_name AS x_organization_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_product_desc AS x_product_desc,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_uom_desc AS x_uom_desc,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_inv_item_flg AS x_inv_item_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_stock_item_flg AS x_stock_item_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_trans_flg AS x_trans_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_rev_flg AS x_rev_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_cost_flg AS x_cost_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_acct AS x_gcoa_acct,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_prod AS x_gcoa_prod,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_tax_cat AS x_tax_cat,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_loc_acct AS x_gcoa_loc_acct,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.datasource_num_id AS datasource_num_id_1,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.row_wid AS row_wid,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.integration_id AS integration_id_1,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.effective_to_dt AS effective_to_dt,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.effective_from_dt AS effective_from_dt_0,\n",
        "                    lkp_w_int_org_d_inventory.effective_from_dt AS effective_from_dt_1,\n",
        "                    lkp_w_int_org_d_inventory.effective_to_dt AS effective_to_dt_1,\n",
        "                    lkp_w_int_org_d_inventory.datasource_num_id AS datasource_num_id_2,\n",
        "                    lkp_w_int_org_d_inventory.integration_id AS integration_id_2,\n",
        "                    lkp_w_int_org_d_inventory.scd1_wid AS scd1_wid,\n",
        "                    lkp_w_product_d_product_wid.effective_from_dt AS effective_from_dt_2,\n",
        "                    lkp_w_product_d_product_wid.effective_to_dt AS effective_to_dt_2,\n",
        "                    lkp_w_product_d_product_wid.datasource_num_id AS datasource_num_id_3,\n",
        "                    lkp_w_product_d_product_wid.integration_id AS integration_id_3,\n",
        "                    lkp_w_product_d_product_wid.scd1_wid AS scd1_wid_1,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.datasource_num_id AS datasource_num_id_4,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.row_wid AS row_wid_1,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.integration_id AS integration_id_4,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.effective_to_dt AS effective_to_dt_3,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.effective_from_dt AS effective_from_dt_3\n",
        "                FROM\n",
        "                    (\n",
        "                        (\n",
        "                            (\n",
        "                                (\n",
        "                                    (\n",
        "                                        (\n",
        "                                            (\n",
        "                                                (\n",
        "                                                    (\n",
        "                                                        (\n",
        "                                                            workspace.prxbi_dw.w_inventory_product_ds AS sq_w_inventory_product_ds_sq_w_in\n",
        "                                                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh1_sq_w_inventory_\n",
        "                                                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat1 = w_prod_cat_dh1_sq_w_inventory_.integration_id\n",
        "                                                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh1_sq_w_inventory_.datasource_num_id\n",
        "                                                        )\n",
        "                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh2_sq_w_inventory_\n",
        "                                                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat2 = w_prod_cat_dh2_sq_w_inventory_.integration_id\n",
        "                                                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh2_sq_w_inventory_.datasource_num_id\n",
        "                                                    )\n",
        "                                                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh3_sq_w_inventory_\n",
        "                                                        ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat3 = w_prod_cat_dh3_sq_w_inventory_.integration_id\n",
        "                                                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh3_sq_w_inventory_.datasource_num_id\n",
        "                                                )\n",
        "                                                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh4_sq_w_inventory_\n",
        "                                                    ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat4 = w_prod_cat_dh4_sq_w_inventory_.integration_id\n",
        "                                                   AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh4_sq_w_inventory_.datasource_num_id\n",
        "                                            )\n",
        "                                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh5_sq_w_inventory_\n",
        "                                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat5 = w_prod_cat_dh5_sq_w_inventory_.integration_id\n",
        "                                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh5_sq_w_inventory_.datasource_num_id\n",
        "                                        )\n",
        "                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh6_sq_w_inventory_\n",
        "                                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat6 = w_prod_cat_dh6_sq_w_inventory_.integration_id\n",
        "                                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh6_sq_w_inventory_.datasource_num_id\n",
        "                                    )\n",
        "                                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh7_sq_w_inventory_\n",
        "                                        ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat7 = w_prod_cat_dh7_sq_w_inventory_.integration_id\n",
        "                                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh7_sq_w_inventory_.datasource_num_id\n",
        "                                )\n",
        "                                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh8_sq_w_inventory_\n",
        "                                    ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat8 = w_prod_cat_dh8_sq_w_inventory_.integration_id\n",
        "                                   AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh8_sq_w_inventory_.datasource_num_id\n",
        "                            )\n",
        "                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh9_sq_w_inventory_\n",
        "                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat9 = w_prod_cat_dh9_sq_w_inventory_.integration_id\n",
        "                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh9_sq_w_inventory_.datasource_num_id\n",
        "                        )\n",
        "                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh10_sq_w_inventory\n",
        "                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat10 = w_prod_cat_dh10_sq_w_inventory.integration_id\n",
        "                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh10_sq_w_inventory.datasource_num_id\n",
        "                    )\n",
        "                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh_unspsc_sq_w_inve\n",
        "                        ON sq_w_inventory_product_ds_sq_w_in.unspsc_code = w_prod_cat_dh_unspsc_sq_w_inve.integration_id\n",
        "                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh_unspsc_sq_w_inve.datasource_num_id\n",
        "                WHERE\n",
        "                    (1 = 1)\n",
        "            ) AS sq_w_inventory_product_ds_sq_w\n",
        "            LEFT OUTER JOIN\n",
        "            (\n",
        "                SELECT\n",
        "                    w_busn_location_d_lkp_w_busn_l.datasource_num_id AS datasource_num_id,\n",
        "                    w_busn_location_d_lkp_w_busn_l.row_wid AS row_wid,\n",
        "                    w_busn_location_d_lkp_w_busn_l.integration_id AS integration_id,\n",
        "                    w_busn_location_d_lkp_w_busn_l.effective_to_dt AS effective_to_dt,\n",
        "                    w_busn_location_d_lkp_w_busn_l.effective_from_dt AS effective_from_dt\n",
        "                FROM\n",
        "                    workspace.prxbi_dw.w_busn_location_d AS w_busn_location_d_lkp_w_busn_l\n",
        "                WHERE\n",
        "                    (1 = 1)\n",
        "            ) AS lkp_w_busn_location_d_lkp_w_bu\n",
        "                ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_busn_location_d_lkp_w_bu.datasource_num_id\n",
        "               AND sq_w_inventory_product_ds_sq_w.plant_loc_id = lkp_w_busn_location_d_lkp_w_bu.integration_id\n",
        "               AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_busn_location_d_lkp_w_bu.effective_from_dt\n",
        "               AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_busn_location_d_lkp_w_bu.effective_to_dt\n",
        "    )\n",
        "    LEFT OUTER JOIN workspace.prxbi_dw.w_int_org_d AS lkp_w_int_org_d_inventory\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_int_org_d_inventory.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.inventory_org_id = lkp_w_int_org_d_inventory.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_int_org_d_inventory.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_int_org_d_inventory.effective_to_dt\n",
        "    LEFT OUTER JOIN workspace.prxbi_dw.w_product_d AS lkp_w_product_d_product_wid\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_product_d_product_wid.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.product_id = lkp_w_product_d_product_wid.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_product_d_product_wid.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_product_d_product_wid.effective_to_dt\n",
        "    LEFT OUTER JOIN\n",
        "    (\n",
        "        SELECT\n",
        "            w_user_d_lkp_w_user_d_changed_.datasource_num_id AS datasource_num_id,\n",
        "            w_user_d_lkp_w_user_d_changed_.row_wid AS row_wid,\n",
        "            w_user_d_lkp_w_user_d_changed_.integration_id AS integration_id,\n",
        "            w_user_d_lkp_w_user_d_changed_.effective_to_dt AS effective_to_dt,\n",
        "            w_user_d_lkp_w_user_d_changed_.effective_from_dt AS effective_from_dt\n",
        "        FROM\n",
        "            workspace.prxbi_dw.w_user_d AS w_user_d_lkp_w_user_d_changed_\n",
        "        WHERE\n",
        "                (1 = 1)\n",
        "            AND (w_user_d_lkp_w_user_d_changed_.delete_flg = 'N')\n",
        "    ) AS lkp_w_user_d_lkp_w_user_d_chan\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_user_d_lkp_w_user_d_chan.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_by_id = lkp_w_user_d_lkp_w_user_d_chan.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_on_dt >= lkp_w_user_d_lkp_w_user_d_chan.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_on_dt < lkp_w_user_d_lkp_w_user_d_chan.effective_to_dt\n",
        "    LEFT OUTER JOIN\n",
        "    (\n",
        "        SELECT\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.datasource_num_id AS datasource_num_id,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.row_wid AS row_wid,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.integration_id AS integration_id,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.effective_to_dt AS effective_to_dt,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.effective_from_dt AS effective_from_dt\n",
        "        FROM\n",
        "            (\n",
        "                SELECT\n",
        "                    w_user_d_lkp_w_user_d_created_.datasource_num_id AS datasource_num_id,\n",
        "                    w_user_d_lkp_w_user_d_created_.row_wid AS row_wid,\n",
        "                    w_user_d_lkp_w_user_d_created_.integration_id AS integration_id,\n",
        "                    w_user_d_lkp_w_user_d_created_.effective_to_dt AS effective_to_dt,\n",
        "                    w_user_d_lkp_w_user_d_created_.effective_from_dt AS effective_from_dt\n",
        "                FROM\n",
        "                    workspace.prxbi_dw.w_user_d AS w_user_d_lkp_w_user_d_created_\n",
        "                WHERE\n",
        "                        (1 = 1)\n",
        "                    AND (w_user_d_lkp_w_user_d_created_.delete_flg = 'N')\n",
        "            ) AS lkp_w_user_d_lkp_w_user_d_crea\n",
        "    ) AS lkp_w_user_d_lkp_w_user_d_cr_1\n",
        "        ON inline_view.datasource_num_id = lkp_w_user_d_lkp_w_user_d_cr_1.datasource_num_id\n",
        "       AND inline_view.created_by_id = lkp_w_user_d_lkp_w_user_d_cr_1.integration_id\n",
        "       AND inline_view.created_on_dt >= lkp_w_user_d_lkp_w_user_d_cr_1.effective_from_dt\n",
        "       AND inline_view.created_on_dt < lkp_w_user_d_lkp_w_user_d_cr_1.effective_to_dt\n",
        "    WHERE (1 = 1)\n",
        ") AS C\n",
        "LEFT OUTER JOIN workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "    ON C.src_eff_from_dt = T.src_eff_from_dt\n",
        "   AND C.datasource_num_id = T.datasource_num_id\n",
        "   AND C.integration_id = T.integration_id;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.i_inventory_product_d_flow;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table Optimization (SCEN_TASK_NO {150})\n",
        "\n",
        "Oracle index creation is not directly applicable to Delta Lake. For performance, ZORDER may be used if this table were persistent and heavily queried. As a temporary flow table, explicit ZORDER is usually not necessary but is included as a best practice example if it were a persistent table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
        "OPTIMIZE workspace.prxbi_dw.i_inventory_product_d_flow ZORDER BY (src_eff_from_dt, datasource_num_id, integration_id);"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error Logging & Other Bypassed Steps (SCEN_TASK_NO {140} - {320})\n",
        "\n",
        "Many ODI tasks related to error logging and specific detection strategies are either bypassed, not applicable, or handled differently in Databricks. These steps have been either removed (PL/SQL blocks, `ALTER SESSION`) or noted as comments where the original ODI task was bypassed or its functionality is inherently different in Delta Lake."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Merge into Target Table (SCEN_TASK_NO {330} - {350})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {330} & {350}: Combined Oracle UPDATE and INSERT into a single MERGE statement\n",
        "MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "USING workspace.prxbi_dw.i_inventory_product_d_flow AS S\n",
        "ON\n",
        "        T.src_eff_from_dt = S.src_eff_from_dt\n",
        "    AND T.datasource_num_id = S.datasource_num_id\n",
        "    AND T.integration_id = S.integration_id\n",
        "WHEN MATCHED AND S.ind_update = 'U' THEN UPDATE SET\n",
        "    T.product_wid = S.product_wid,\n",
        "    T.inventory_org_wid = S.inventory_org_wid,\n",
        "    T.plant_loc_wid = S.plant_loc_wid,\n",
        "    T.product_num = S.product_num,\n",
        "    T.abc_ind = S.abc_ind,\n",
        "    T.planner_code = S.planner_code,\n",
        "    T.procurement_type_code = S.procurement_type_code,\n",
        "    T.spc_proc_type_code = S.spc_proc_type_code,\n",
        "    T.buyer_code = S.buyer_code,\n",
        "    T.buyer_name = S.buyer_name,\n",
        "    T.commodity_code = S.commodity_code,\n",
        "    T.commodity_uom_code = S.commodity_uom_code,\n",
        "    T.profit_center_num = S.profit_center_num,\n",
        "    T.reorder_point = S.reorder_point,\n",
        "    T.safety_stock_level = S.safety_stock_level,\n",
        "    T.min_lot_size = S.min_lot_size,\n",
        "    T.max_lot_size = S.max_lot_size,\n",
        "    T.fixed_lot_size = S.fixed_lot_size,\n",
        "    T.max_stock_level = S.max_stock_level,\n",
        "    T.lot_ordering_cost = S.lot_ordering_cost,\n",
        "    T.mrp_time_fence = S.mrp_time_fence,\n",
        "    T.ext_procure_time = S.ext_procure_time,\n",
        "    T.internal_mfg_time = S.internal_mfg_time,\n",
        "    T.max_storage_days = S.max_storage_days,\n",
        "    T.mrp_profile_code = S.mrp_profile_code,\n",
        "    T.mrp_type_code = S.mrp_type_code,\n",
        "    T.mrp_grp_code = S.mrp_grp_code,\n",
        "    T.lot_size_code = S.lot_size_code,\n",
        "    T.backflush_ind = S.backflush_ind,\n",
        "    T.qa_inspect_ind = S.qa_inspect_ind,\n",
        "    T.repetitive_mfg_ind = S.repetitive_mfg_ind,\n",
        "    T.bulk_item_ind = S.bulk_item_ind,\n",
        "    T.forecast_period = S.forecast_period,\n",
        "    T.mfg_uom_code = S.mfg_uom_code,\n",
        "    T.issue_uom_code = S.issue_uom_code,\n",
        "    T.manufacturing_place = S.manufacturing_place,\n",
        "    T.loading_type_code = S.loading_type_code,\n",
        "    T.int_store_loc_code = S.int_store_loc_code,\n",
        "    T.ext_store_loc_code = S.ext_store_loc_code,\n",
        "    T.active_flg = S.active_flg,\n",
        "    T.created_by_wid = S.created_by_wid,\n",
        "    T.changed_by_wid = S.changed_by_wid,\n",
        "    T.created_on_dt = S.created_on_dt,\n",
        "    T.changed_on_dt = S.changed_on_dt,\n",
        "    T.aux1_changed_on_dt = S.aux1_changed_on_dt,\n",
        "    T.aux2_changed_on_dt = S.aux2_changed_on_dt,\n",
        "    T.aux3_changed_on_dt = S.aux3_changed_on_dt,\n",
        "    T.aux4_changed_on_dt = S.aux4_changed_on_dt,\n",
        "    T.src_eff_to_dt = S.src_eff_to_dt,\n",
        "    T.effective_from_dt = S.effective_from_dt,\n",
        "    T.delete_flg = S.delete_flg,\n",
        "    T.tenant_id = S.tenant_id,\n",
        "    T.x_custom = S.x_custom,\n",
        "    T.inv_prod_cat1 = S.inv_prod_cat1,\n",
        "    T.inv_prod_cat2 = S.inv_prod_cat2,\n",
        "    T.inv_prod_cat3 = S.inv_prod_cat3,\n",
        "    T.inv_prod_cat4 = S.inv_prod_cat4,\n",
        "    T.inv_prod_cat5 = S.inv_prod_cat5,\n",
        "    T.inv_prod_cat6 = S.inv_prod_cat6,\n",
        "    T.inv_prod_cat7 = S.inv_prod_cat7,\n",
        "    T.inv_prod_cat8 = S.inv_prod_cat8,\n",
        "    T.inv_prod_cat9 = S.inv_prod_cat9,\n",
        "    T.inv_prod_cat10 = S.inv_prod_cat10,\n",
        "    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid,\n",
        "    T.inv_prod_cat2_wid = S.inv_prod_cat2_wid,\n",
        "    T.inv_prod_cat3_wid = S.inv_prod_cat3_wid,\n",
        "    T.inv_prod_cat4_wid = S.inv_prod_cat4_wid,\n",
        "    T.inv_prod_cat5_wid = S.inv_prod_cat5_wid,\n",
        "    T.inv_prod_cat6_wid = S.inv_prod_cat6_wid,\n",
        "    T.inv_prod_cat7_wid = S.inv_prod_cat7_wid,\n",
        "    T.inv_prod_cat8_wid = S.inv_prod_cat8_wid,\n",
        "    T.inv_prod_cat9_wid = S.inv_prod_cat9_wid,\n",
        "    T.inv_prod_cat10_wid = S.inv_prod_cat10_wid,\n",
        "    T.invoiceable_item_flag = S.invoiceable_item_flag,\n",
        "    T.invoice_enabled_flag = S.invoice_enabled_flag,\n",
        "    T.primary_uom_code = S.primary_uom_code,\n",
        "    T.c_primary_uom_code = S.c_primary_uom_code,\n",
        "    T.unspsc_code = S.unspsc_code,\n",
        "    T.unspsc_inv_prod_cat_wid = S.unspsc_inv_prod_cat_wid,\n",
        "    T.commodity_name = S.commodity_name,\n",
        "    T.commodity_uom_name = S.commodity_uom_name,\n",
        "    T.ext_store_loc_name = S.ext_store_loc_name,\n",
        "    T.int_store_loc_name = S.int_store_loc_name,\n",
        "    T.issue_uom_name = S.issue_uom_name,\n",
        "    T.loading_type_name = S.loading_type_name,\n",
        "    T.lot_size_name = S.lot_size_name,\n",
        "    T.mfg_uom_name = S.mfg_uom_name,\n",
        "    T.mrp_grp_name = S.mrp_grp_name,\n",
        "    T.mrp_profile_name = S.mrp_profile_name,\n",
        "    T.mrp_type_name = S.mrp_type_name,\n",
        "    T.planner_name = S.planner_name,\n",
        "    T.primary_uom_name = S.primary_uom_name,\n",
        "    T.procurement_type_name = S.procurement_type_name,\n",
        "    T.profit_center_name = S.profit_center_name,\n",
        "    T.spc_proc_type_name = S.spc_proc_type_name,\n",
        "    T.status_code = S.status_code,\n",
        "    T.w_status_code = S.w_status_code,\n",
        "    T.product_type_code = S.product_type_code,\n",
        "    T.make_buy_ind = S.make_buy_ind,\n",
        "    T.fixed_lead_time = S.fixed_lead_time,\n",
        "    T.variable_lead_time = S.variable_lead_time,\n",
        "    T.cumulative_total_lead_time = S.cumulative_total_lead_time,\n",
        "    T.postprocessing_lead_time = S.postprocessing_lead_time,\n",
        "    T.preprocessing_lead_time = S.preprocessing_lead_time,\n",
        "    T.process_quality_enabled_flg = S.process_quality_enabled_flg,\n",
        "    T.x_price_sequence = S.x_price_sequence,\n",
        "    T.x_organization_name = S.x_organization_name,\n",
        "    T.x_product_desc = S.x_product_desc,\n",
        "    T.x_uom_desc = S.x_uom_desc,\n",
        "    T.x_inv_item_flg = S.x_inv_item_flg,\n",
        "    T.x_stock_item_flg = S.x_stock_item_flg,\n",
        "    T.x_trans_flg = S.x_trans_flg,\n",
        "    T.x_rev_flg = S.x_rev_flg,\n",
        "    T.x_cost_flg = S.x_cost_flg,\n",
        "    T.x_gcoa_acct = S.x_gcoa_acct,\n",
        "    T.x_gcoa_prod = S.x_gcoa_prod,\n",
        "    T.x_tax_cat = S.x_tax_cat,\n",
        "    T.organization_id = S.organization_id,\n",
        "    T.x_gcoa_loc_acct = S.x_gcoa_loc_acct,\n",
        "    T.effective_to_dt = to_timestamp('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss'),\n",
        "    T.current_flg = 'Y',\n",
        "    T.w_update_dt = current_timestamp(),\n",
        "    T.etl_proc_wid = ${ETL_PROC_WID}\n",
        "WHEN NOT MATCHED AND S.ind_update = 'I' THEN INSERT (\n",
        "    product_wid,\n",
        "    inventory_org_wid,\n",
        "    plant_loc_wid,\n",
        "    product_num,\n",
        "    abc_ind,\n",
        "    planner_code,\n",
        "    procurement_type_code,\n",
        "    spc_proc_type_code,\n",
        "    buyer_code,\n",
        "    buyer_name,\n",
        "    commodity_code,\n",
        "    commodity_uom_code,\n",
        "    profit_center_num,\n",
        "    reorder_point,\n",
        "    safety_stock_level,\n",
        "    min_lot_size,\n",
        "    max_lot_size,\n",
        "    fixed_lot_size,\n",
        "    max_stock_level,\n",
        "    lot_ordering_cost,\n",
        "    mrp_time_fence,\n",
        "    ext_procure_time,\n",
        "    internal_mfg_time,\n",
        "    max_storage_days,\n",
        "    mrp_profile_code,\n",
        "    mrp_type_code,\n",
        "    mrp_grp_code,\n",
        "    lot_size_code,\n",
        "    backflush_ind,\n",
        "    qa_inspect_ind,\n",
        "    repetitive_mfg_ind,\n",
        "    bulk_item_ind,\n",
        "    forecast_period,\n",
        "    mfg_uom_code,\n",
        "    issue_uom_code,\n",
        "    manufacturing_place,\n",
        "    loading_type_code,\n",
        "    int_store_loc_code,\n",
        "    ext_store_loc_code,\n",
        "    active_flg,\n",
        "    created_by_wid,\n",
        "    changed_by_wid,\n",
        "    created_on_dt,\n",
        "    changed_on_dt,\n",
        "    aux1_changed_on_dt,\n",
        "    aux2_changed_on_dt,\n",
        "    aux3_changed_on_dt,\n",
        "    aux4_changed_on_dt,\n",
        "    src_eff_from_dt,\n",
        "    src_eff_to_dt,\n",
        "    effective_from_dt,\n",
        "    delete_flg,\n",
        "    datasource_num_id,\n",
        "    integration_id,\n",
        "    tenant_id,\n",
        "    x_custom,\n",
        "    inv_prod_cat1,\n",
        "    inv_prod_cat2,\n",
        "    inv_prod_cat3,\n",
        "    inv_prod_cat4,\n",
        "    inv_prod_cat5,\n",
        "    inv_prod_cat6,\n",
        "    inv_prod_cat7,\n",
        "    inv_prod_cat8,\n",
        "    inv_prod_cat9,\n",
        "    inv_prod_cat10,\n",
        "    inv_prod_cat1_wid,\n",
        "    inv_prod_cat2_wid,\n",
        "    inv_prod_cat3_wid,\n",
        "    inv_prod_cat4_wid,\n",
        "    inv_prod_cat5_wid,\n",
        "    inv_prod_cat6_wid,\n",
        "    inv_prod_cat7_wid,\n",
        "    inv_prod_cat8_wid,\n",
        "    inv_prod_cat9_wid,\n",
        "    inv_prod_cat10_wid,\n",
        "    invoiceable_item_flag,\n",
        "    invoice_enabled_flag,\n",
        "    primary_uom_code,\n",
        "    c_primary_uom_code,\n",
        "    unspsc_code,\n",
        "    unspsc_inv_prod_cat_wid,\n",
        "    commodity_name,\n",
        "    commodity_uom_name,\n",
        "    ext_store_loc_name,\n",
        "    int_store_loc_name,\n",
        "    issue_uom_name,\n",
        "    loading_type_name,\n",
        "    lot_size_name,\n",
        "    mfg_uom_name,\n",
        "    mrp_grp_name,\n",
        "    mrp_profile_name,\n",
        "    mrp_type_name,\n",
        "    planner_name,\n",
        "    primary_uom_name,\n",
        "    procurement_type_name,\n",
        "    profit_center_name,\n",
        "    spc_proc_type_name,\n",
        "    status_code,\n",
        "    w_status_code,\n",
        "    product_type_code,\n",
        "    make_buy_ind,\n",
        "    fixed_lead_time,\n",
        "    variable_lead_time,\n",
        "    cumulative_total_lead_time,\n",
        "    postprocessing_lead_time,\n",
        "    preprocessing_lead_time,\n",
        "    process_quality_enabled_flg,\n",
        "    x_price_sequence,\n",
        "    x_organization_name,\n",
        "    x_product_desc,\n",
        "    x_uom_desc,\n",
        "    x_inv_item_flg,\n",
        "    x_stock_item_flg,\n",
        "    x_trans_flg,\n",
        "    x_rev_flg,\n",
        "    x_cost_flg,\n",
        "    x_gcoa_acct,\n",
        "    x_gcoa_prod,\n",
        "    x_tax_cat,\n",
        "    organization_id,\n",
        "    x_gcoa_loc_acct,\n",
        "    row_wid,\n",
        "    w_insert_dt,\n",
        "    w_update_dt,\n",
        "    etl_proc_wid,\n",
        "    current_flg,\n",
        "    effective_to_dt\n",
        ") VALUES (\n",
        "    S.product_wid,\n",
        "    S.inventory_org_wid,\n",
        "    S.plant_loc_wid,\n",
        "    S.product_num,\n",
        "    S.abc_ind,\n",
        "    S.planner_code,\n",
        "    S.procurement_type_code,\n",
        "    S.spc_proc_type_code,\n",
        "    S.buyer_code,\n",
        "    S.buyer_name,\n",
        "    S.commodity_code,\n",
        "    S.commodity_uom_code,\n",
        "    S.profit_center_num,\n",
        "    S.reorder_point,\n",
        "    S.safety_stock_level,\n",
        "    S.min_lot_size,\n",
        "    S.max_lot_size,\n",
        "    S.fixed_lot_size,\n",
        "    S.max_stock_level,\n",
        "    S.lot_ordering_cost,\n",
        "    S.mrp_time_fence,\n",
        "    S.ext_procure_time,\n",
        "    S.internal_mfg_time,\n",
        "    S.max_storage_days,\n",
        "    S.mrp_profile_code,\n",
        "    S.mrp_type_code,\n",
        "    S.mrp_grp_code,\n",
        "    S.lot_size_code,\n",
        "    S.backflush_ind,\n",
        "    S.qa_inspect_ind,\n",
        "    S.repetitive_mfg_ind,\n",
        "    S.bulk_item_ind,\n",
        "    S.forecast_period,\n",
        "    S.mfg_uom_code,\n",
        "    S.issue_uom_code,\n",
        "    S.manufacturing_place,\n",
        "    S.loading_type_code,\n",
        "    S.int_store_loc_code,\n",
        "    S.ext_store_loc_code,\n",
        "    S.active_flg,\n",
        "    S.created_by_wid,\n",
        "    S.changed_by_wid,\n",
        "    S.created_on_dt,\n",
        "    S.changed_on_dt,\n",
        "    S.aux1_changed_on_dt,\n",
        "    S.aux2_changed_on_dt,\n",
        "    S.aux3_changed_on_dt,\n",
        "    S.aux4_changed_on_dt,\n",
        "    S.src_eff_from_dt,\n",
        "    S.src_eff_to_dt,\n",
        "    S.effective_from_dt,\n",
        "    S.delete_flg,\n",
        "    S.datasource_num_id,\n",
        "    S.integration_id,\n",
        "    S.tenant_id,\n",
        "    S.x_custom,\n",
        "    S.inv_prod_cat1,\n",
        "    S.inv_prod_cat2,\n",
        "    S.inv_prod_cat3,\n",
        "    S.inv_prod_cat4,\n",
        "    S.inv_prod_cat5,\n",
        "    S.inv_prod_cat6,\n",
        "    S.inv_prod_cat7,\n",
        "    S.inv_prod_cat8,\n",
        "    S.inv_prod_cat9,\n",
        "    S.inv_prod_cat10,\n",
        "    S.inv_prod_cat1_wid,\n",
        "    S.inv_prod_cat2_wid,\n",
        "    S.inv_prod_cat3_wid,\n",
        "    S.inv_prod_cat4_wid,\n",
        "    S.inv_prod_cat5_wid,\n",
        "    S.inv_prod_cat6_wid,\n",
        "    S.inv_prod_cat7_wid,\n",
        "    S.inv_prod_cat8_wid,\n",
        "    S.inv_prod_cat9_wid,\n",
        "    S.inv_prod_cat10_wid,\n",
        "    S.invoiceable_item_flag,\n",
        "    S.invoice_enabled_flag,\n",
        "    S.primary_uom_code,\n",
        "    S.c_primary_uom_code,\n",
        "    S.unspsc_code,\n",
        "    S.unspsc_inv_prod_cat_wid,\n",
        "    S.commodity_name,\n",
        "    S.commodity_uom_name,\n",
        "    S.ext_store_loc_name,\n",
        "    S.int_store_loc_name,\n",
        "    S.issue_uom_name,\n",
        "    S.loading_type_name,\n",
        "    S.lot_size_name,\n",
        "    S.mfg_uom_name,\n",
        "    S.mrp_grp_name,\n",
        "    S.mrp_profile_name,\n",
        "    S.mrp_type_name,\n",
        "    S.planner_name,\n",
        "    S.primary_uom_name,\n",
        "    S.procurement_type_name,\n",
        "    S.profit_center_name,\n",
        "    S.spc_proc_type_name,\n",
        "    S.status_code,\n",
        "    S.w_status_code,\n",
        "    S.product_type_code,\n",
        "    S.make_buy_ind,\n",
        "    S.fixed_lead_time,\n",
        "    S.variable_lead_time,\n",
        "    S.cumulative_total_lead_time,\n",
        "    S.postprocessing_lead_time,\n",
        "    S.preprocessing_lead_time,\n",
        "    S.process_quality_enabled_flg,\n",
        "    S.x_price_sequence,\n",
        "    S.x_organization_name,\n",
        "    S.x_product_desc,\n",
        "    S.x_uom_desc,\n",
        "    S.x_inv_item_flg,\n",
        "    S.x_stock_item_flg,\n",
        "    S.x_trans_flg,\n",
        "    S.x_rev_flg,\n",
        "    S.x_cost_flg,\n",
        "    S.x_gcoa_acct,\n",
        "    S.x_gcoa_prod,\n",
        "    S.x_tax_cat,\n",
        "    S.organization_id,\n",
        "    S.x_gcoa_loc_acct,\n",
        "    CAST(monotonically_increasing_id() AS BIGINT),\n",
        "    current_timestamp(),\n",
        "    current_timestamp(),\n",
        "    ${ETL_PROC_WID},\n",
        "    S.current_flg,\n",
        "    S.effective_to_dt\n",
        ");"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Update ETL Load Dates (SCEN_TASK_NO {360} - {370})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {360}: Update reference dates\n",
        "UPDATE workspace.prxbi_dw.w_etl_load_dates\n",
        "SET\n",
        "    target_table_name = 'W_INVENTORY_PRODUCT_D',\n",
        "    etl_proc_wid = ${ETL_PROC_WID},\n",
        "    load_plan_id = ${EXECUTION_ID},\n",
        "    wip_load_start_date = date_sub(current_date(), ${PRUNE_DAYS}),\n",
        "    etl_load_date = current_timestamp(),\n",
        "    committed = CASE WHEN '${IS_INCREMENTAL}' = 'Y' THEN '1' ELSE '0' END\n",
        "WHERE\n",
        "        datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "    AND package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "    AND etl_usage_code = '${ETL_USAGE_CODE}';"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {370}: Insert reference dates into History Table\n",
        "INSERT INTO workspace.prxbi_dw.w_etl_load_dates_log\n",
        "(\n",
        "    datasource_num_id,\n",
        "    package_name,\n",
        "    target_table_name,\n",
        "    etl_usage_code,\n",
        "    etl_proc_wid,\n",
        "    load_plan_id,\n",
        "    session_id,\n",
        "    wip_load_start_date,\n",
        "    last_max_date,\n",
        "    etl_load_date,\n",
        "    committed\n",
        ")\n",
        "SELECT\n",
        "    datasource_num_id,\n",
        "    package_name,\n",
        "    target_table_name,\n",
        "    etl_usage_code,\n",
        "    etl_proc_wid,\n",
        "    load_plan_id,\n",
        "    ${ODI_SESS_NO},\n",
        "    wip_load_start_date,\n",
        "    last_max_date,\n",
        "    etl_load_date,\n",
        "    committed\n",
        "FROM\n",
        "    workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE\n",
        "        datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "    AND package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "    AND etl_usage_code = '${ETL_USAGE_CODE}';"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Cleanup (SCEN_TASK_NO {380} - {440})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {380}: COMMIT - Implicit in Databricks Delta transactions, removed.\n",
        "-- SCEN_TASK_NO {390}: Table Truncate behaviour - Step bypassed.\n",
        "-- SCEN_TASK_NO {400}: Blank task.\n",
        "-- SCEN_TASK_NO {410}: Blank task.\n",
        "\n",
        "-- SCEN_TASK_NO {420}: Optionally drops flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {430}: Step bypassed.\n",
        "-- SCEN_TASK_NO {440}: Conditional error table drop is not performed as per Databricks E$ table handling rules.\n",
        "-- E$ tables are persistent and records are deleted by session ID, not dropped."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.w_inventory_product_d;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Conversion Notes and Manual Actions Required\n",
        "\n",
        "1.  **ODI Parameters:** Ensure all `dbutils.widgets.text` calls for ODI parameters (`DATASOURCE_NUM_ID`, `WH_DATASOURCE_NUM_ID`, `ETL_USAGE_CODE`, `LOW_DATE`, `SOURCE_CODE`, `TARGET_CODE`, `ETL_PROC_WID`, `EXECUTION_ID`, `PRUNE_DAYS`, `IS_INCREMENTAL`, `ODI_SESS_NO`) are correctly configured with default values or populated at runtime.\n",
        "2.  **Oracle `ROW_WID` (Sequence):** The `W_INVENTORY_PRODUCT_D_SEQ.NEXTVAL` for `ROW_WID` in the INSERT statement has been replaced with `CAST(monotonically_increasing_id() AS BIGINT)`. Verify if `ROW_WID` in the target table `w_inventory_product_d` is truly a surrogate key or if it has a business meaning requiring a specific generation logic.\n",
        "3.  **Error Table Handling:** The Oracle logic for dropping the error table (`E$_...`) conditionally has been removed. In Databricks, error tables (`e_inventory_product_d`) are generally persistent, and records are typically deleted by `ODI_SESS_NO` rather than dropping the table. A `CREATE TABLE IF NOT EXISTS` is used to ensure the error table exists.\n",
        "4.  **Oracle-specific SQL:** `ALTER SESSION`, `BEGIN...END;` blocks, `/*+ append */` hints, `NOLOGGING`, `PURGE` clauses, and Oracle `CREATE INDEX` statements have been removed or translated to their Databricks equivalents.\n",
        "5.  **Data Type Mapping:** Oracle `NUMBER` types have been mapped to `BIGINT` or `STRING` in the flow and error tables based on context (e.g., `NUMBER(10,0)` to `BIGINT`, generic `NUMBER` to `BIGINT`, or to `STRING` if used to store raw, potentially non-numeric error values). `VARCHAR2` and `CHAR` are mapped to `STRING`, `DATE` and `TIMESTAMP(n)` to `TIMESTAMP`, `UROWID` to `STRING`.\n",
        "6.  **`TO_DATE`/`TO_TIMESTAMP` Format Strings:** All Oracle date/timestamp format strings have been converted to Spark SQL compatible format strings (e.g., `YYYY-MM-DD HH24:MI:SS` -> `yyyy-MM-dd HH:mm:ss`).\n",
        "7.  **`ROWID` in `CASE` statement:** The use of `T.ROWID IS NOT NULL` in the `IND_UPDATE` logic has been replaced with `T.integration_id IS NOT NULL` which serves the same purpose of checking for a match in the target table based on its primary key.\n",
        "8.  **ZORDER:** An `OPTIMIZE ... ZORDER BY` statement for the flow table has been included as a best practice, but for temporary flow tables, it might not be strictly necessary as they are short-lived. The `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` is included as a mandatory guard."
      ]
    }
  ]
}
```
<end_of_output>```json
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Spark SQL Conversion\n",
        "\n",
        "**Source File:** `SILOS_SIL_INVENTORYPRODUCTDIMENSION.txt`\n",
        "**Conversion Timestamp:** `2024-07-30T12:00:00Z`\n",
        "\n",
        "This notebook contains the converted Spark SQL logic from the original ODI session. It performs incremental updates to the `W_INVENTORY_PRODUCT_D` dimension table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"WH_DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_USAGE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"LOW_DATE\", \"1900-01-01 00:00:00\")\n",
        "dbutils.widgets.text(\"SOURCE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"TARGET_CODE\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"EXECUTION_ID\", \"\")\n",
        "dbutils.widgets.text(\"PRUNE_DAYS\", \"0\")\n",
        "dbutils.widgets.text(\"IS_INCREMENTAL\", \"Y\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_check_load_status AS\n",
        "SELECT\n",
        "    CASE\n",
        "        WHEN COUNT(*) > 0 THEN 'Y'\n",
        "        ELSE 'N'\n",
        "    END AS LOAD_STATUS\n",
        "FROM\n",
        "    workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE\n",
        "    package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "    AND (datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "         OR datasource_num_id = ${WH_DATASOURCE_NUM_ID})\n",
        "    AND etl_usage_code = '${ETL_USAGE_CODE}'\n",
        "    AND committed = '1';"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_low_date AS\n",
        "SELECT to_timestamp('${LOW_DATE}', 'yyyy-MM-dd HH:mm:ss') AS etl_low_date;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "display(spark.sql(\"SELECT * FROM v_check_load_status;\"))\n",
        "display(spark.sql(\"SELECT * FROM v_etl_low_date;\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Category Lookup Update (SCEN_TASK_NO {2})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "USING (\n",
        "    SELECT DISTINCT\n",
        "        X.integration_id,\n",
        "        X.inv_prod_cat1,\n",
        "        Y.inv_prod_cat1_wid\n",
        "    FROM\n",
        "        (\n",
        "            SELECT\n",
        "                B.integration_id,\n",
        "                A.integration_id AS inv_prod_cat1\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp AS A,\n",
        "                workspace.prxbi_dw.w_inventory_product_d AS B\n",
        "            WHERE\n",
        "                    CONCAT_WS('~', A.inventory_item_id, A.organization_id) = B.integration_id\n",
        "                AND A.integration_id <> B.inv_prod_cat1\n",
        "        ) AS X,\n",
        "        (\n",
        "            SELECT\n",
        "                P.integration_id,\n",
        "                Q.row_wid AS inv_prod_cat1_wid\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp AS P,\n",
        "                workspace.prxbi_dw.w_prod_cat_dh AS Q\n",
        "            WHERE\n",
        "                Q.integration_id = P.integration_id\n",
        "        ) AS Y\n",
        "    WHERE\n",
        "        X.inv_prod_cat1 = Y.integration_id\n",
        ") AS S\n",
        "ON (T.integration_id = S.integration_id)\n",
        "WHEN MATCHED THEN UPDATE SET\n",
        "    T.inv_prod_cat1 = S.inv_prod_cat1,\n",
        "    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Metadata and Initialization (SCEN_TASK_NO {10} - {50})\n",
        "\n",
        "Oracle PL/SQL blocks and session settings are not applicable to Databricks Spark SQL and have been removed."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error Table (SCEN_TASK_NO {60} - {70})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {60}: Drop error table (Oracle purge removed, E$ tables are typically persistent)\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.e_inventory_product_d;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {70}: Creates the error table\n",
        "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_inventory_product_d\n",
        "(\n",
        "    ora_err_number        BIGINT,\n",
        "    ora_err_mesg          STRING,\n",
        "    ora_err_rowid         STRING,\n",
        "    ora_err_optyp         STRING,\n",
        "    ora_err_tag           STRING,\n",
        "    ind_update            STRING,\n",
        "    diagnostic_rowid      STRING,\n",
        "    error_type_ind        STRING,\n",
        "    autocorrect_ind       STRING  DEFAULT 'N',\n",
        "    autocorrect_code      STRING,\n",
        "    autocorrect_desc      STRING,\n",
        "    committed             STRING DEFAULT '0',\n",
        "    row_wid               STRING,\n",
        "    product_wid           STRING,\n",
        "    inventory_org_wid     STRING,\n",
        "    plant_loc_wid         STRING,\n",
        "    product_num           STRING,\n",
        "    abc_ind               STRING,\n",
        "    planner_code          STRING,\n",
        "    procurement_type_code STRING,\n",
        "    spc_proc_type_code    STRING,\n",
        "    buyer_code            STRING,\n",
        "    buyer_name            STRING,\n",
        "    commodity_code        STRING,\n",
        "    commodity_uom_code    STRING,\n",
        "    profit_center_num     STRING,\n",
        "    reorder_point         STRING,\n",
        "    safety_stock_level    STRING,\n",
        "    min_lot_size          STRING,\n",
        "    max_lot_size          STRING,\n",
        "    fixed_lot_size        STRING,\n",
        "    max_stock_level       STRING,\n",
        "    lot_ordering_cost     STRING,\n",
        "    mrp_time_fence        STRING,\n",
        "    ext_procure_time      STRING,\n",
        "    internal_mfg_time     STRING,\n",
        "    max_storage_days      STRING,\n",
        "    mrp_profile_code      STRING,\n",
        "    mrp_type_code         STRING,\n",
        "    mrp_grp_code          STRING,\n",
        "    lot_size_code         STRING,\n",
        "    backflush_ind         STRING,\n",
        "    qa_inspect_ind        STRING,\n",
        "    repetitive_mfg_ind    STRING,\n",
        "    bulk_item_ind         STRING,\n",
        "    forecast_period       STRING,\n",
        "    mfg_uom_code          STRING,\n",
        "    issue_uom_code        STRING,\n",
        "    manufacturing_place   STRING,\n",
        "    loading_type_code     STRING,\n",
        "    int_store_loc_code    STRING,\n",
        "    ext_store_loc_code    STRING,\n",
        "    active_flg            STRING,\n",
        "    created_by_wid        STRING,\n",
        "    changed_by_wid        STRING,\n",
        "    created_on_dt         STRING,\n",
        "    changed_on_dt         STRING,\n",
        "    aux1_changed_on_dt    STRING,\n",
        "    aux2_changed_on_dt    STRING,\n",
        "    aux3_changed_on_dt    STRING,\n",
        "    aux4_changed_on_dt    STRING,\n",
        "    src_eff_from_dt       STRING,\n",
        "    src_eff_to_dt         STRING,\n",
        "    effective_from_dt     STRING,\n",
        "    effective_to_dt       STRING,\n",
        "    current_flg           STRING,\n",
        "    w_insert_dt           STRING,\n",
        "    w_update_dt           STRING,\n",
        "    datasource_num_id     STRING,\n",
        "    etl_proc_wid          STRING,\n",
        "    integration_id        STRING,\n",
        "    tenant_id             STRING,\n",
        "    x_custom              STRING,\n",
        "    inv_prod_cat1         STRING,\n",
        "    inv_prod_cat2         STRING,\n",
        "    inv_prod_cat3         STRING,\n",
        "    inv_prod_cat4         STRING,\n",
        "    inv_prod_cat5         STRING,\n",
        "    inv_prod_cat6         STRING,\n",
        "    inv_prod_cat7         STRING,\n",
        "    inv_prod_cat8         STRING,\n",
        "    inv_prod_cat9         STRING,\n",
        "    inv_prod_cat10        STRING,\n",
        "    inv_prod_cat1_wid     STRING,\n",
        "    inv_prod_cat2_wid     STRING,\n",
        "    inv_prod_cat3_wid     STRING,\n",
        "    inv_prod_cat4_wid     STRING,\n",
        "    inv_prod_cat5_wid     STRING,\n",
        "    inv_prod_cat6_wid     STRING,\n",
        "    inv_prod_cat7_wid     STRING,\n",
        "    inv_prod_cat8_wid     STRING,\n",
        "    inv_prod_cat9_wid     STRING,\n",
        "    inv_prod_cat10_wid    STRING,\n",
        "    invoiceable_item_flag STRING,\n",
        "    invoice_enabled_flag  STRING,\n",
        "    primary_uom_code      STRING,\n",
        "    c_primary_uom_code    STRING,\n",
        "    unspsc_code           STRING,\n",
        "    unspsc_inv_prod_cat_wid STRING,\n",
        "    commodity_name        STRING,\n",
        "    commodity_uom_name    STRING,\n",
        "    ext_store_loc_name    STRING,\n",
        "    int_store_loc_name    STRING,\n",
        "    issue_uom_name        STRING,\n",
        "    loading_type_name     STRING,\n",
        "    lot_size_name         STRING,\n",
        "    mfg_uom_name          STRING,\n",
        "    mrp_grp_name          STRING,\n",
        "    mrp_profile_name      STRING,\n",
        "    mrp_type_name         STRING,\n",
        "    planner_name          STRING,\n",
        "    primary_uom_name      STRING,\n",
        "    procurement_type_name STRING,\n",
        "    profit_center_name    STRING,\n",
        "    spc_proc_type_name    STRING,\n",
        "    status_code           STRING,\n",
        "    w_status_code         STRING,\n",
        "    product_type_code     STRING,\n",
        "    make_buy_ind          STRING,\n",
        "    fixed_lead_time       STRING,\n",
        "    variable_lead_time    STRING,\n",
        "    cumulative_total_lead_time STRING,\n",
        "    preprocessing_lead_time    STRING,\n",
        "    postprocessing_lead_time   STRING,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence      STRING,\n",
        "    x_organization_name   STRING,\n",
        "    x_product_desc        STRING,\n",
        "    x_uom_desc            STRING,\n",
        "    x_inv_item_flg        STRING,\n",
        "    x_stock_item_flg      STRING,\n",
        "    x_trans_flg           STRING,\n",
        "    x_rev_flg             STRING,\n",
        "    x_cost_flg            STRING,\n",
        "    x_gcoa_acct           STRING,\n",
        "    x_gcoa_prod           STRING,\n",
        "    x_tax_cat             STRING,\n",
        "    organization_id       STRING,\n",
        "    x_gcoa_loc_acct       STRING,\n",
        "    delete_flg            STRING\n",
        ")\n",
        "USING DELTA;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table (SCEN_TASK_NO {110} - {130})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {110}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {120}: Creates the flow table\n",
        "CREATE TABLE workspace.prxbi_dw.i_inventory_product_d_flow\n",
        "(\n",
        "    src_eff_from_dt       TIMESTAMP,\n",
        "    datasource_num_id     BIGINT,\n",
        "    integration_id        STRING,\n",
        "    row_wid               BIGINT,\n",
        "    product_wid           BIGINT,\n",
        "    inventory_org_wid     BIGINT,\n",
        "    plant_loc_wid         BIGINT,\n",
        "    product_num           STRING,\n",
        "    abc_ind               STRING,\n",
        "    planner_code          STRING,\n",
        "    procurement_type_code STRING,\n",
        "    spc_proc_type_code    STRING,\n",
        "    buyer_code            STRING,\n",
        "    buyer_name            STRING,\n",
        "    commodity_code        STRING,\n",
        "    commodity_uom_code    STRING,\n",
        "    profit_center_num     STRING,\n",
        "    reorder_point         BIGINT,\n",
        "    safety_stock_level    BIGINT,\n",
        "    min_lot_size          BIGINT,\n",
        "    max_lot_size          BIGINT,\n",
        "    fixed_lot_size        BIGINT,\n",
        "    max_stock_level       BIGINT,\n",
        "    lot_ordering_cost     BIGINT,\n",
        "    mrp_time_fence        BIGINT,\n",
        "    ext_procure_time      BIGINT,\n",
        "    internal_mfg_time     BIGINT,\n",
        "    max_storage_days      BIGINT,\n",
        "    mrp_profile_code      STRING,\n",
        "    mrp_type_code         STRING,\n",
        "    mrp_grp_code          STRING,\n",
        "    lot_size_code         STRING,\n",
        "    backflush_ind         STRING,\n",
        "    qa_inspect_ind        STRING,\n",
        "    repetitive_mfg_ind    STRING,\n",
        "    bulk_item_ind         STRING,\n",
        "    forecast_period       STRING,\n",
        "    mfg_uom_code          STRING,\n",
        "    issue_uom_code        STRING,\n",
        "    manufacturing_place   STRING,\n",
        "    loading_type_code     STRING,\n",
        "    int_store_loc_code    STRING,\n",
        "    ext_store_loc_code    STRING,\n",
        "    active_flg            STRING,\n",
        "    created_by_wid        BIGINT,\n",
        "    changed_by_wid        BIGINT,\n",
        "    created_on_dt         TIMESTAMP,\n",
        "    changed_on_dt         TIMESTAMP,\n",
        "    aux1_changed_on_dt    TIMESTAMP,\n",
        "    aux2_changed_on_dt    TIMESTAMP,\n",
        "    aux3_changed_on_dt    TIMESTAMP,\n",
        "    aux4_changed_on_dt    TIMESTAMP,\n",
        "    src_eff_to_dt         TIMESTAMP,\n",
        "    effective_from_dt     TIMESTAMP,\n",
        "    effective_to_dt       TIMESTAMP,\n",
        "    delete_flg            STRING,\n",
        "    current_flg           STRING,\n",
        "    w_insert_dt           TIMESTAMP,\n",
        "    w_update_dt           TIMESTAMP,\n",
        "    etl_proc_wid          BIGINT,\n",
        "    tenant_id             STRING,\n",
        "    x_custom              STRING,\n",
        "    inv_prod_cat1         STRING,\n",
        "    inv_prod_cat2         STRING,\n",
        "    inv_prod_cat3         STRING,\n",
        "    inv_prod_cat4         STRING,\n",
        "    inv_prod_cat5         STRING,\n",
        "    inv_prod_cat6         STRING,\n",
        "    inv_prod_cat7         STRING,\n",
        "    inv_prod_cat8         STRING,\n",
        "    inv_prod_cat9         STRING,\n",
        "    inv_prod_cat10        STRING,\n",
        "    inv_prod_cat1_wid     BIGINT,\n",
        "    inv_prod_cat2_wid     BIGINT,\n",
        "    inv_prod_cat3_wid     BIGINT,\n",
        "    inv_prod_cat4_wid     BIGINT,\n",
        "    inv_prod_cat5_wid     BIGINT,\n",
        "    inv_prod_cat6_wid     BIGINT,\n",
        "    inv_prod_cat7_wid     BIGINT,\n",
        "    inv_prod_cat8_wid     BIGINT,\n",
        "    inv_prod_cat9_wid     BIGINT,\n",
        "    inv_prod_cat10_wid    BIGINT,\n",
        "    invoiceable_item_flag STRING,\n",
        "    invoice_enabled_flag  STRING,\n",
        "    primary_uom_code      STRING,\n",
        "    c_primary_uom_code    STRING,\n",
        "    unspsc_code           STRING,\n",
        "    unspsc_inv_prod_cat_wid BIGINT,\n",
        "    commodity_name        STRING,\n",
        "    commodity_uom_name    STRING,\n",
        "    ext_store_loc_name    STRING,\n",
        "    int_store_loc_name    STRING,\n",
        "    issue_uom_name        STRING,\n",
        "    loading_type_name     STRING,\n",
        "    lot_size_name         STRING,\n",
        "    mfg_uom_name          STRING,\n",
        "    mrp_grp_name          STRING,\n",
        "    mrp_profile_name      STRING,\n",
        "    mrp_type_name         STRING,\n",
        "    planner_name          STRING,\n",
        "    primary_uom_name      STRING,\n",
        "    procurement_type_name STRING,\n",
        "    profit_center_name    STRING,\n",
        "    spc_proc_type_name    STRING,\n",
        "    status_code           STRING,\n",
        "    w_status_code         STRING,\n",
        "    product_type_code     STRING,\n",
        "    make_buy_ind          STRING,\n",
        "    fixed_lead_time       BIGINT,\n",
        "    variable_lead_time    BIGINT,\n",
        "    cumulative_total_lead_time BIGINT,\n",
        "    postprocessing_lead_time BIGINT,\n",
        "    preprocessing_lead_time BIGINT,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence      STRING,\n",
        "    x_organization_name   STRING,\n",
        "    x_product_desc        STRING,\n",
        "    x_uom_desc            STRING,\n",
        "    x_inv_item_flg        STRING,\n",
        "    x_stock_item_flg      STRING,\n",
        "    x_trans_flg           STRING,\n",
        "    x_rev_flg             STRING,\n",
        "    x_cost_flg            STRING,\n",
        "    x_gcoa_acct           STRING,\n",
        "    x_gcoa_prod           STRING,\n",
        "    x_tax_cat             STRING,\n",
        "    organization_id       STRING,\n",
        "    x_gcoa_loc_acct       STRING,\n",
        "    ind_update            STRING\n",
        ")\n",
        "USING DELTA;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {130}: Insert into flow table\n",
        "INSERT INTO workspace.prxbi_dw.i_inventory_product_d_flow\n",
        "(\n",
        "    product_wid,\n",
        "    inventory_org_wid,\n",
        "    plant_loc_wid,\n",
        "    product_num,\n",
        "    abc_ind,\n",
        "    planner_code,\n",
        "    procurement_type_code,\n",
        "    spc_proc_type_code,\n",
        "    buyer_code,\n",
        "    buyer_name,\n",
        "    commodity_code,\n",
        "    commodity_uom_code,\n",
        "    profit_center_num,\n",
        "    reorder_point,\n",
        "    safety_stock_level,\n",
        "    min_lot_size,\n",
        "    max_lot_size,\n",
        "    fixed_lot_size,\n",
        "    max_stock_level,\n",
        "    lot_ordering_cost,\n",
        "    mrp_time_fence,\n",
        "    ext_procure_time,\n",
        "    internal_mfg_time,\n",
        "    max_storage_days,\n",
        "    mrp_profile_code,\n",
        "    mrp_type_code,\n",
        "    mrp_grp_code,\n",
        "    lot_size_code,\n",
        "    backflush_ind,\n",
        "    qa_inspect_ind,\n",
        "    repetitive_mfg_ind,\n",
        "    bulk_item_ind,\n",
        "    forecast_period,\n",
        "    mfg_uom_code,\n",
        "    issue_uom_code,\n",
        "    manufacturing_place,\n",
        "    loading_type_code,\n",
        "    int_store_loc_code,\n",
        "    ext_store_loc_code,\n",
        "    active_flg,\n",
        "    created_by_wid,\n",
        "    changed_by_wid,\n",
        "    created_on_dt,\n",
        "    changed_on_dt,\n",
        "    aux1_changed_on_dt,\n",
        "    aux2_changed_on_dt,\n",
        "    aux3_changed_on_dt,\n",
        "    aux4_changed_on_dt,\n",
        "    src_eff_from_dt,\n",
        "    src_eff_to_dt,\n",
        "    effective_from_dt,\n",
        "    delete_flg,\n",
        "    datasource_num_id,\n",
        "    integration_id,\n",
        "    tenant_id,\n",
        "    x_custom,\n",
        "    inv_prod_cat1,\n",
        "    inv_prod_cat2,\n",
        "    inv_prod_cat3,\n",
        "    inv_prod_cat4,\n",
        "    inv_prod_cat5,\n",
        "    inv_prod_cat6,\n",
        "    inv_prod_cat7,\n",
        "    inv_prod_cat8,\n",
        "    inv_prod_cat9,\n",
        "    inv_prod_cat10,\n",
        "    inv_prod_cat1_wid,\n",
        "    inv_prod_cat2_wid,\n",
        "    inv_prod_cat3_wid,\n",
        "    inv_prod_cat4_wid,\n",
        "    inv_prod_cat5_wid,\n",
        "    inv_prod_cat6_wid,\n",
        "    inv_prod_cat7_wid,\n",
        "    inv_prod_cat8_wid,\n",
        "    inv_prod_cat9_wid,\n",
        "    inv_prod_cat10_wid,\n",
        "    invoiceable_item_flag,\n",
        "    invoice_enabled_flag,\n",
        "    primary_uom_code,\n",
        "    c_primary_uom_code,\n",
        "    unspsc_code,\n",
        "    unspsc_inv_prod_cat_wid,\n",
        "    commodity_name,\n",
        "    commodity_uom_name,\n",
        "    ext_store_loc_name,\n",
        "    int_store_loc_name,\n",
        "    issue_uom_name,\n",
        "    loading_type_name,\n",
        "    lot_size_name,\n",
        "    mfg_uom_name,\n",
        "    mrp_grp_name,\n",
        "    mrp_profile_name,\n",
        "    mrp_type_name,\n",
        "    planner_name,\n",
        "    primary_uom_name,\n",
        "    procurement_type_name,\n",
        "    profit_center_name,\n",
        "    spc_proc_type_name,\n",
        "    status_code,\n",
        "    w_status_code,\n",
        "    product_type_code,\n",
        "    make_buy_ind,\n",
        "    fixed_lead_time,\n",
        "    variable_lead_time,\n",
        "    cumulative_total_lead_time,\n",
        "    postprocessing_lead_time,\n",
        "    preprocessing_lead_time,\n",
        "    process_quality_enabled_flg,\n",
        "    x_price_sequence,\n",
        "    x_organization_name,\n",
        "    x_product_desc,\n",
        "    x_uom_desc,\n",
        "    x_inv_item_flg,\n",
        "    x_stock_item_flg,\n",
        "    x_trans_flg,\n",
        "    x_rev_flg,\n",
        "    x_cost_flg,\n",
        "    x_gcoa_acct,\n",
        "    x_gcoa_prod,\n",
        "    x_tax_cat,\n",
        "    organization_id,\n",
        "    x_gcoa_loc_acct,\n",
        "    current_flg,\n",
        "    effective_to_dt,\n",
        "    ind_update\n",
        ")\n",
        "SELECT\n",
        "    C.product_wid,\n",
        "    C.inventory_org_wid,\n",
        "    C.plant_loc_wid,\n",
        "    C.product_num,\n",
        "    C.abc_ind,\n",
        "    C.planner_code,\n",
        "    C.procurement_type_code,\n",
        "    C.spc_proc_type_code,\n",
        "    C.buyer_code,\n",
        "    C.buyer_name,\n",
        "    C.commodity_code,\n",
        "    C.commodity_uom_code,\n",
        "    C.profit_center_num,\n",
        "    C.reorder_point,\n",
        "    C.safety_stock_level,\n",
        "    C.min_lot_size,\n",
        "    C.max_lot_size,\n",
        "    C.fixed_lot_size,\n",
        "    C.max_stock_level,\n",
        "    C.lot_ordering_cost,\n",
        "    C.mrp_time_fence,\n",
        "    C.ext_procure_time,\n",
        "    C.internal_mfg_time,\n",
        "    C.max_storage_days,\n",
        "    C.mrp_profile_code,\n",
        "    C.mrp_type_code,\n",
        "    C.mrp_grp_code,\n",
        "    C.lot_size_code,\n",
        "    C.backflush_ind,\n",
        "    C.qa_inspect_ind,\n",
        "    C.repetitive_mfg_ind,\n",
        "    C.bulk_item_ind,\n",
        "    C.forecast_period,\n",
        "    C.mfg_uom_code,\n",
        "    C.issue_uom_code,\n",
        "    C.manufacturing_place,\n",
        "    C.loading_type_code,\n",
        "    C.int_store_loc_code,\n",
        "    C.ext_store_loc_code,\n",
        "    C.active_flg,\n",
        "    C.created_by_wid,\n",
        "    C.changed_by_wid,\n",
        "    C.created_on_dt,\n",
        "    C.changed_on_dt,\n",
        "    C.aux1_changed_on_dt,\n",
        "    C.aux2_changed_on_dt,\n",
        "    C.aux3_changed_on_dt,\n",
        "    C.aux4_changed_on_dt,\n",
        "    C.src_eff_from_dt,\n",
        "    C.src_eff_to_dt,\n",
        "    C.effective_from_dt,\n",
        "    C.delete_flg,\n",
        "    C.datasource_num_id,\n",
        "    C.integration_id,\n",
        "    C.tenant_id,\n",
        "    C.x_custom,\n",
        "    C.inv_prod_cat1,\n",
        "    C.inv_prod_cat2,\n",
        "    C.inv_prod_cat3,\n",
        "    C.inv_prod_cat4,\n",
        "    C.inv_prod_cat5,\n",
        "    C.inv_prod_cat6,\n",
        "    C.inv_prod_cat7,\n",
        "    C.inv_prod_cat8,\n",
        "    C.inv_prod_cat9,\n",
        "    C.inv_prod_cat10,\n",
        "    C.inv_prod_cat1_wid,\n",
        "    C.inv_prod_cat2_wid,\n",
        "    C.inv_prod_cat3_wid,\n",
        "    C.inv_prod_cat4_wid,\n",
        "    C.inv_prod_cat5_wid,\n",
        "    C.inv_prod_cat6_wid,\n",
        "    C.inv_prod_cat7_wid,\n",
        "    C.inv_prod_cat8_wid,\n",
        "    C.inv_prod_cat9_wid,\n",
        "    C.inv_prod_cat10_wid,\n",
        "    C.invoiceable_item_flag,\n",
        "    C.invoice_enabled_flag,\n",
        "    C.primary_uom_code,\n",
        "    C.c_primary_uom_code,\n",
        "    C.unspsc_code,\n",
        "    C.unspsc_inv_prod_cat_wid,\n",
        "    C.commodity_name,\n",
        "    C.commodity_uom_name,\n",
        "    C.ext_store_loc_name,\n",
        "    C.int_store_loc_name,\n",
        "    C.issue_uom_name,\n",
        "    C.loading_type_name,\n",
        "    C.lot_size_name,\n",
        "    C.mfg_uom_name,\n",
        "    C.mrp_grp_name,\n",
        "    C.mrp_profile_name,\n",
        "    C.mrp_type_name,\n",
        "    C.planner_name,\n",
        "    C.primary_uom_name,\n",
        "    C.procurement_type_name,\n",
        "    C.profit_center_name,\n",
        "    C.spc_proc_type_name,\n",
        "    C.status_code,\n",
        "    C.w_status_code,\n",
        "    C.product_type_code,\n",
        "    C.make_buy_ind,\n",
        "    C.fixed_lead_time,\n",
        "    C.variable_lead_time,\n",
        "    C.cumulative_total_lead_time,\n",
        "    C.postprocessing_lead_time,\n",
        "    C.preprocessing_lead_time,\n",
        "    C.process_quality_enabled_flg,\n",
        "    C.x_price_sequence,\n",
        "    C.x_organization_name,\n",
        "    C.x_product_desc,\n",
        "    C.x_uom_desc,\n",
        "    C.x_inv_item_flg,\n",
        "    C.x_stock_item_flg,\n",
        "    C.x_trans_flg,\n",
        "    C.x_rev_flg,\n",
        "    C.x_cost_flg,\n",
        "    C.x_gcoa_acct,\n",
        "    C.x_gcoa_prod,\n",
        "    C.x_tax_cat,\n",
        "    C.organization_id,\n",
        "    C.x_gcoa_loc_acct,\n",
        "    'Y' AS current_flg,\n",
        "    to_timestamp('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss') AS effective_to_dt,\n",
        "    CASE\n",
        "        WHEN T.integration_id IS NOT NULL\n",
        "             AND (\n",
        "                 T.changed_on_dt = C.changed_on_dt\n",
        "                 OR (T.changed_on_dt IS NULL AND C.changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux1_changed_on_dt = C.aux1_changed_on_dt\n",
        "                 OR (T.aux1_changed_on_dt IS NULL AND C.aux1_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux2_changed_on_dt = C.aux2_changed_on_dt\n",
        "                 OR (T.aux2_changed_on_dt IS NULL AND C.aux2_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux3_changed_on_dt = C.aux3_changed_on_dt\n",
        "                 OR (T.aux3_changed_on_dt IS NULL AND C.aux3_changed_on_dt IS NULL)\n",
        "             )\n",
        "             AND (\n",
        "                 T.aux4_changed_on_dt = C.aux4_changed_on_dt\n",
        "                 OR (T.aux4_changed_on_dt IS NULL AND C.aux4_changed_on_dt IS NULL)\n",
        "             )\n",
        "        THEN 'N'\n",
        "        WHEN T.integration_id IS NOT NULL THEN 'U'\n",
        "        ELSE 'I'\n",
        "    END AS ind_update\n",
        "FROM\n",
        "    (\n",
        "        SELECT\n",
        "            COALESCE(inline_view.scd1_wid_1, 0) AS product_wid,\n",
        "            COALESCE(inline_view.scd1_wid, 0) AS inventory_org_wid,\n",
        "            COALESCE(inline_view.row_wid, 0) AS plant_loc_wid,\n",
        "            inline_view.product_num AS product_num,\n",
        "            inline_view.abc_ind AS abc_ind,\n",
        "            COALESCE(inline_view.planner_code, '__NOT_APPLICABLE__') AS planner_code,\n",
        "            COALESCE(inline_view.procurement_type_code, '__NOT_APPLICABLE__') AS procurement_type_code,\n",
        "            COALESCE(inline_view.spc_proc_type_code, '__NOT_APPLICABLE__') AS spc_proc_type_code,\n",
        "            COALESCE(inline_view.buyer_code, '__NOT_APPLICABLE__') AS buyer_code,\n",
        "            inline_view.buyer_name AS buyer_name,\n",
        "            COALESCE(inline_view.commodity_code, '__NOT_APPLICABLE__') AS commodity_code,\n",
        "            COALESCE(inline_view.commodity_uom_code, '__NOT_APPLICABLE__') AS commodity_uom_code,\n",
        "            inline_view.profit_center_num AS profit_center_num,\n",
        "            inline_view.reorder_point AS reorder_point,\n",
        "            inline_view.safety_stock_level AS safety_stock_level,\n",
        "            inline_view.min_lot_size AS min_lot_size,\n",
        "            inline_view.max_lot_size AS max_lot_size,\n",
        "            inline_view.fixed_lot_size AS fixed_lot_size,\n",
        "            inline_view.max_stock_level AS max_stock_level,\n",
        "            inline_view.lot_ordering_cost AS lot_ordering_cost,\n",
        "            inline_view.mrp_time_fence AS mrp_time_fence,\n",
        "            inline_view.ext_procure_time AS ext_procure_time,\n",
        "            inline_view.internal_mfg_time AS internal_mfg_time,\n",
        "            inline_view.max_storage_days AS max_storage_days,\n",
        "            COALESCE(inline_view.mrp_profile_code, '__NOT_APPLICABLE__') AS mrp_profile_code,\n",
        "            COALESCE(inline_view.mrp_type_code, '__NOT_APPLICABLE__') AS mrp_type_code,\n",
        "            COALESCE(inline_view.mrp_grp_code, '__NOT_APPLICABLE__') AS mrp_grp_code,\n",
        "            COALESCE(inline_view.lot_size_code, '__NOT_APPLICABLE__') AS lot_size_code,\n",
        "            inline_view.backflush_ind AS backflush_ind,\n",
        "            inline_view.qa_inspect_ind AS qa_inspect_ind,\n",
        "            inline_view.repetitive_mfg_ind AS repetitive_mfg_ind,\n",
        "            inline_view.bulk_item_ind AS bulk_item_ind,\n",
        "            inline_view.forecast_period AS forecast_period,\n",
        "            COALESCE(inline_view.mfg_uom_code, '__NOT_APPLICABLE__') AS mfg_uom_code,\n",
        "            COALESCE(inline_view.issue_uom_code, '__NOT_APPLICABLE__') AS issue_uom_code,\n",
        "            inline_view.manufacturing_place AS manufacturing_place,\n",
        "            COALESCE(inline_view.loading_type_code, '__NOT_APPLICABLE__') AS loading_type_code,\n",
        "            COALESCE(inline_view.int_store_loc_code, '__NOT_APPLICABLE__') AS int_store_loc_code,\n",
        "            COALESCE(inline_view.ext_store_loc_code, '__NOT_APPLICABLE__') AS ext_store_loc_code,\n",
        "            inline_view.active_flg AS active_flg,\n",
        "            COALESCE(lkp_w_user_d_lkp_w_user_d_cr_1.row_wid, 0) AS created_by_wid,\n",
        "            COALESCE(inline_view.row_wid_1, 0) AS changed_by_wid,\n",
        "            inline_view.created_on_dt AS created_on_dt,\n",
        "            inline_view.changed_on_dt AS changed_on_dt,\n",
        "            inline_view.aux1_changed_on_dt AS aux1_changed_on_dt,\n",
        "            inline_view.aux2_changed_on_dt AS aux2_changed_on_dt,\n",
        "            inline_view.aux3_changed_on_dt AS aux3_changed_on_dt,\n",
        "            inline_view.aux4_changed_on_dt AS aux4_changed_on_dt,\n",
        "            inline_view.src_eff_from_dt AS src_eff_from_dt,\n",
        "            inline_view.src_eff_to_dt AS src_eff_to_dt,\n",
        "            COALESCE(inline_view.src_eff_from_dt, to_timestamp('${LOW_DATE}', 'yyyy-MM-dd HH:mm:ss')) AS effective_from_dt,\n",
        "            (CASE WHEN inline_view.delete_flg = 'Y' THEN 'Y' ELSE 'N' END) AS delete_flg,\n",
        "            inline_view.datasource_num_id AS datasource_num_id,\n",
        "            inline_view.integration_id AS integration_id,\n",
        "            inline_view.tenant_id AS tenant_id,\n",
        "            inline_view.x_custom AS x_custom,\n",
        "            inline_view.inv_prod_cat1 AS inv_prod_cat1,\n",
        "            inline_view.inv_prod_cat2 AS inv_prod_cat2,\n",
        "            inline_view.inv_prod_cat3 AS inv_prod_cat3,\n",
        "            inline_view.inv_prod_cat4 AS inv_prod_cat4,\n",
        "            inline_view.inv_prod_cat5 AS inv_prod_cat5,\n",
        "            inline_view.inv_prod_cat6 AS inv_prod_cat6,\n",
        "            inline_view.inv_prod_cat7 AS inv_prod_cat7,\n",
        "            inline_view.inv_prod_cat8 AS inv_prod_cat8,\n",
        "            inline_view.inv_prod_cat9 AS inv_prod_cat9,\n",
        "            inline_view.inv_prod_cat10 AS inv_prod_cat10,\n",
        "            (CASE WHEN inline_view.inv_prod_cat1 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat1_row_wid, 0) END) AS inv_prod_cat1_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat2 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat2_row_wid, 0) END) AS inv_prod_cat2_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat3 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat3_row_wid, 0) END) AS inv_prod_cat3_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat4 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat4_row_wid, 0) END) AS inv_prod_cat4_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat5 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat5_row_wid, 0) END) AS inv_prod_cat5_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat6 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat6_row_wid, 0) END) AS inv_prod_cat6_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat7 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat7_row_wid, 0) END) AS inv_prod_cat7_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat8 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat8_row_wid, 0) END) AS inv_prod_cat8_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat9 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat9_row_wid, 0) END) AS inv_prod_cat9_wid,\n",
        "            (CASE WHEN inline_view.inv_prod_cat10 IS NULL THEN 0 ELSE COALESCE(inline_view.inv_prod_cat10_row_wid, 0) END) AS inv_prod_cat10_wid,\n",
        "            COALESCE(inline_view.invoiceable_item_flag, 'N') AS invoiceable_item_flag,\n",
        "            COALESCE(inline_view.invoice_enabled_flag, 'N') AS invoice_enabled_flag,\n",
        "            COALESCE(inline_view.primary_uom_code, '__NOT_APPLICABLE__') AS primary_uom_code,\n",
        "            COALESCE(\n",
        "                (SELECT T1.trg_domain_member_code FROM workspace.prxbi_dw.w_domain_member_map_g AS T1\n",
        "                 WHERE T1.src_domain_code = '${SOURCE_CODE}'\n",
        "                   AND T1.src_domain_member_code = COALESCE(inline_view.primary_uom_code, '__UNASSIGNED__')\n",
        "                   AND T1.src_datasource_num_id IN (inline_view.datasource_num_id, 999)\n",
        "                   AND T1.trg_domain_code = '${TARGET_CODE}'),\n",
        "                (SELECT T2.trg_domain_member_code FROM workspace.prxbi_dw.w_domain_member_map_g AS T2\n",
        "                 WHERE T2.src_domain_code = '${SOURCE_CODE}'\n",
        "                   AND T2.src_domain_member_code = '__ANY__'\n",
        "                   AND T2.src_datasource_num_id IN (inline_view.datasource_num_id, 999)\n",
        "                   AND T2.trg_domain_code = '${TARGET_CODE}'),\n",
        "                (CASE\n",
        "                    WHEN inline_view.primary_uom_code IS NULL THEN COALESCE(\n",
        "                        (SELECT T3.domain_member_code FROM workspace.prxbi_dw.w_domain_member_g AS T3\n",
        "                         WHERE T3.domain_member_code = '__UNASSIGNED__'\n",
        "                           AND T3.domain_code = '${TARGET_CODE}'),\n",
        "                        '__ERROR__'\n",
        "                    )\n",
        "                    ELSE (CASE WHEN '${TARGET_CODE}' = 'W_LANGUAGE' THEN '_ERR' ELSE '__ERROR__' END)\n",
        "                END)\n",
        "            ) AS c_primary_uom_code,\n",
        "            COALESCE(inline_view.unspsc_code, '__NOT_APPLICABLE__') AS unspsc_code,\n",
        "            COALESCE(inline_view.inv_prod_cat_unspsc_row_wid, 0) AS unspsc_inv_prod_cat_wid,\n",
        "            inline_view.commodity_name AS commodity_name,\n",
        "            inline_view.commodity_uom_name AS commodity_uom_name,\n",
        "            inline_view.ext_store_loc_name AS ext_store_loc_name,\n",
        "            inline_view.int_store_loc_name AS int_store_loc_name,\n",
        "            inline_view.issue_uom_name AS issue_uom_name,\n",
        "            inline_view.loading_type_name AS loading_type_name,\n",
        "            inline_view.lot_size_name AS lot_size_name,\n",
        "            inline_view.mfg_uom_name AS mfg_uom_name,\n",
        "            inline_view.mrp_grp_name AS mrp_grp_name,\n",
        "            inline_view.mrp_profile_name AS mrp_profile_name,\n",
        "            inline_view.mrp_type_name AS mrp_type_name,\n",
        "            inline_view.planner_name AS planner_name,\n",
        "            inline_view.primary_uom_name AS primary_uom_name,\n",
        "            inline_view.procurement_type_name AS procurement_type_name,\n",
        "            inline_view.profit_center_name AS profit_center_name,\n",
        "            inline_view.spc_proc_type_name AS spc_proc_type_name,\n",
        "            inline_view.status_code AS status_code,\n",
        "            inline_view.w_status_code AS w_status_code,\n",
        "            inline_view.product_type_code AS product_type_code,\n",
        "            inline_view.make_buy_ind AS make_buy_ind,\n",
        "            inline_view.fixed_lead_time AS fixed_lead_time,\n",
        "            inline_view.variable_lead_time AS variable_lead_time,\n",
        "            inline_view.cumulative_total_lead_time AS cumulative_total_lead_time,\n",
        "            inline_view.preprocessing_lead_time AS postprocessing_lead_time,\n",
        "            inline_view.preprocessing_lead_time AS preprocessing_lead_time,\n",
        "            inline_view.process_quality_enabled_flg AS process_quality_enabled_flg,\n",
        "            inline_view.x_price_sequence AS x_price_sequence,\n",
        "            inline_view.x_organization_name AS x_organization_name,\n",
        "            inline_view.x_product_desc AS x_product_desc,\n",
        "            inline_view.x_uom_desc AS x_uom_desc,\n",
        "            inline_view.x_inv_item_flg AS x_inv_item_flg,\n",
        "            inline_view.x_stock_item_flg AS x_stock_item_flg,\n",
        "            inline_view.x_trans_flg AS x_trans_flg,\n",
        "            inline_view.x_rev_flg AS x_rev_flg,\n",
        "            inline_view.x_cost_flg AS x_cost_flg,\n",
        "            inline_view.x_gcoa_acct AS x_gcoa_acct,\n",
        "            inline_view.x_gcoa_prod AS x_gcoa_prod,\n",
        "            inline_view.x_tax_cat AS x_tax_cat,\n",
        "            inline_view.inventory_org_id AS organization_id,\n",
        "            inline_view.x_gcoa_loc_acct AS x_gcoa_loc_acct\n",
        "        FROM\n",
        "            (\n",
        "                SELECT\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_type_name AS mrp_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_custom AS x_custom,\n",
        "                    sq_w_inventory_product_ds_sq_w.src_eff_to_dt AS src_eff_to_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_procure_time AS ext_procure_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_uom_code AS commodity_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat10 AS inv_prod_cat10,\n",
        "                    sq_w_inventory_product_ds_sq_w.invoice_enabled_flag AS invoice_enabled_flag,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_profile_code AS mrp_profile_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_store_loc_name AS ext_store_loc_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux3_changed_on_dt AS aux3_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.min_lot_size AS min_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.reorder_point AS reorder_point,\n",
        "                    sq_w_inventory_product_ds_sq_w.bulk_item_ind AS bulk_item_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.spc_proc_type_name AS spc_proc_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_code AS commodity_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.repetitive_mfg_ind AS repetitive_mfg_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.loading_type_code AS loading_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.profit_center_name AS profit_center_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.created_by_id AS created_by_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.buyer_code AS buyer_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.int_store_loc_name AS int_store_loc_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_name AS commodity_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.planner_name AS planner_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_storage_days AS max_storage_days,\n",
        "                    sq_w_inventory_product_ds_sq_w.planner_code AS planner_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_lot_size AS max_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_grp_code AS mrp_grp_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_ordering_cost AS lot_ordering_cost,\n",
        "                    sq_w_inventory_product_ds_sq_w.internal_mfg_time AS internal_mfg_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.manufacturing_place AS manufacturing_place,\n",
        "                    sq_w_inventory_product_ds_sq_w.procurement_type_name AS procurement_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.plant_loc_id AS plant_loc_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.changed_by_id AS changed_by_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.backflush_ind AS backflush_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux2_changed_on_dt AS aux2_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.datasource_num_id AS datasource_num_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.spc_proc_type_code AS spc_proc_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_size_code AS lot_size_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.changed_on_dt AS changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_id AS product_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.primary_uom_code AS primary_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_num AS product_num,\n",
        "                    sq_w_inventory_product_ds_sq_w.forecast_period AS forecast_period,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux1_changed_on_dt AS aux1_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.buyer_name AS buyer_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.fixed_lot_size AS fixed_lot_size,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_grp_name AS mrp_grp_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.procurement_type_code AS procurement_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.primary_uom_name AS primary_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.max_stock_level AS max_stock_level,\n",
        "                    sq_w_inventory_product_ds_sq_w.safety_stock_level AS safety_stock_level,\n",
        "                    sq_w_inventory_product_ds_sq_w.int_store_loc_code AS int_store_loc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.ext_store_loc_code AS ext_store_loc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.issue_uom_name AS issue_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.abc_ind AS abc_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.profit_center_num AS profit_center_num,\n",
        "                    sq_w_inventory_product_ds_sq_w.created_on_dt AS created_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.unspsc_code AS unspsc_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat1 AS inv_prod_cat1,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat3 AS inv_prod_cat3,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat2 AS inv_prod_cat2,\n",
        "                    sq_w_inventory_product_ds_sq_w.commodity_uom_name AS commodity_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat5 AS inv_prod_cat5,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat4 AS inv_prod_cat4,\n",
        "                    sq_w_inventory_product_ds_sq_w.mfg_uom_name AS mfg_uom_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat7 AS inv_prod_cat7,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat6 AS inv_prod_cat6,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat9 AS inv_prod_cat9,\n",
        "                    sq_w_inventory_product_ds_sq_w.inv_prod_cat8 AS inv_prod_cat8,\n",
        "                    sq_w_inventory_product_ds_sq_w.aux4_changed_on_dt AS aux4_changed_on_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.lot_size_name AS lot_size_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.tenant_id AS tenant_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.invoiceable_item_flag AS invoiceable_item_flag,\n",
        "                    sq_w_inventory_product_ds_sq_w.inventory_org_id AS inventory_org_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.src_eff_from_dt AS src_eff_from_dt,\n",
        "                    sq_w_inventory_product_ds_sq_w.integration_id AS integration_id,\n",
        "                    sq_w_inventory_product_ds_sq_w.issue_uom_code AS issue_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.delete_flg AS delete_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_type_code AS mrp_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.active_flg AS active_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_time_fence AS mrp_time_fence,\n",
        "                    sq_w_inventory_product_ds_sq_w.qa_inspect_ind AS qa_inspect_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.mfg_uom_code AS mfg_uom_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.loading_type_name AS loading_type_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.mrp_profile_name AS mrp_profile_name,\n",
        "                    w_prod_cat_dh1_sq_w_inventory_.row_wid AS inv_prod_cat1_row_wid,\n",
        "                    w_prod_cat_dh2_sq_w_inventory_.row_wid AS inv_prod_cat2_row_wid,\n",
        "                    w_prod_cat_dh3_sq_w_inventory_.row_wid AS inv_prod_cat3_row_wid,\n",
        "                    w_prod_cat_dh4_sq_w_inventory_.row_wid AS inv_prod_cat4_row_wid,\n",
        "                    w_prod_cat_dh5_sq_w_inventory_.row_wid AS inv_prod_cat5_row_wid,\n",
        "                    w_prod_cat_dh6_sq_w_inventory_.row_wid AS inv_prod_cat6_row_wid,\n",
        "                    w_prod_cat_dh7_sq_w_inventory_.row_wid AS inv_prod_cat7_row_wid,\n",
        "                    w_prod_cat_dh8_sq_w_inventory_.row_wid AS inv_prod_cat8_row_wid,\n",
        "                    w_prod_cat_dh_unspsc_sq_w_inve.row_wid AS inv_prod_cat_unspsc_row_wid,\n",
        "                    w_prod_cat_dh9_sq_w_inventory_.row_wid AS inv_prod_cat9_row_wid,\n",
        "                    w_prod_cat_dh10_sq_w_inventory.row_wid AS inv_prod_cat10_row_wid,\n",
        "                    sq_w_inventory_product_ds_sq_w.cumulative_total_lead_time AS cumulative_total_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.fixed_lead_time AS fixed_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.w_status_code AS w_status_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.preprocessing_lead_time AS preprocessing_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.status_code AS status_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.variable_lead_time AS variable_lead_time,\n",
        "                    sq_w_inventory_product_ds_sq_w.make_buy_ind AS make_buy_ind,\n",
        "                    sq_w_inventory_product_ds_sq_w.process_quality_enabled_flg AS process_quality_enabled_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.product_type_code AS product_type_code,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_price_sequence AS x_price_sequence,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_organization_name AS x_organization_name,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_product_desc AS x_product_desc,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_uom_desc AS x_uom_desc,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_inv_item_flg AS x_inv_item_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_stock_item_flg AS x_stock_item_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_trans_flg AS x_trans_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_rev_flg AS x_rev_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_cost_flg AS x_cost_flg,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_acct AS x_gcoa_acct,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_prod AS x_gcoa_prod,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_tax_cat AS x_tax_cat,\n",
        "                    sq_w_inventory_product_ds_sq_w.x_gcoa_loc_acct AS x_gcoa_loc_acct,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.datasource_num_id AS datasource_num_id_1,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.row_wid AS row_wid,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.integration_id AS integration_id_1,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.effective_to_dt AS effective_to_dt,\n",
        "                    lkp_w_busn_location_d_lkp_w_bu.effective_from_dt AS effective_from_dt_0,\n",
        "                    lkp_w_int_org_d_inventory.effective_from_dt AS effective_from_dt_1,\n",
        "                    lkp_w_int_org_d_inventory.effective_to_dt AS effective_to_dt_1,\n",
        "                    lkp_w_int_org_d_inventory.datasource_num_id AS datasource_num_id_2,\n",
        "                    lkp_w_int_org_d_inventory.integration_id AS integration_id_2,\n",
        "                    lkp_w_int_org_d_inventory.scd1_wid AS scd1_wid,\n",
        "                    lkp_w_product_d_product_wid.effective_from_dt AS effective_from_dt_2,\n",
        "                    lkp_w_product_d_product_wid.effective_to_dt AS effective_to_dt_2,\n",
        "                    lkp_w_product_d_product_wid.datasource_num_id AS datasource_num_id_3,\n",
        "                    lkp_w_product_d_product_wid.integration_id AS integration_id_3,\n",
        "                    lkp_w_product_d_product_wid.scd1_wid AS scd1_wid_1,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.datasource_num_id AS datasource_num_id_4,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.row_wid AS row_wid_1,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.integration_id AS integration_id_4,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.effective_to_dt AS effective_to_dt_3,\n",
        "                    lkp_w_user_d_lkp_w_user_d_chan.effective_from_dt AS effective_from_dt_3\n",
        "                FROM\n",
        "                    (\n",
        "                        (\n",
        "                            (\n",
        "                                (\n",
        "                                    (\n",
        "                                        (\n",
        "                                            (\n",
        "                                                (\n",
        "                                                    (\n",
        "                                                        (\n",
        "                                                            workspace.prxbi_dw.w_inventory_product_ds AS sq_w_inventory_product_ds_sq_w_in\n",
        "                                                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh1_sq_w_inventory_\n",
        "                                                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat1 = w_prod_cat_dh1_sq_w_inventory_.integration_id\n",
        "                                                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh1_sq_w_inventory_.datasource_num_id\n",
        "                                                        )\n",
        "                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh2_sq_w_inventory_\n",
        "                                                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat2 = w_prod_cat_dh2_sq_w_inventory_.integration_id\n",
        "                                                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh2_sq_w_inventory_.datasource_num_id\n",
        "                                                    )\n",
        "                                                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh3_sq_w_inventory_\n",
        "                                                        ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat3 = w_prod_cat_dh3_sq_w_inventory_.integration_id\n",
        "                                                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh3_sq_w_inventory_.datasource_num_id\n",
        "                                                )\n",
        "                                                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh4_sq_w_inventory_\n",
        "                                                    ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat4 = w_prod_cat_dh4_sq_w_inventory_.integration_id\n",
        "                                                   AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh4_sq_w_inventory_.datasource_num_id\n",
        "                                            )\n",
        "                                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh5_sq_w_inventory_\n",
        "                                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat5 = w_prod_cat_dh5_sq_w_inventory_.integration_id\n",
        "                                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh5_sq_w_inventory_.datasource_num_id\n",
        "                                        )\n",
        "                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh6_sq_w_inventory_\n",
        "                                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat6 = w_prod_cat_dh6_sq_w_inventory_.integration_id\n",
        "                                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh6_sq_w_inventory_.datasource_num_id\n",
        "                                    )\n",
        "                                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh7_sq_w_inventory_\n",
        "                                        ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat7 = w_prod_cat_dh7_sq_w_inventory_.integration_id\n",
        "                                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh7_sq_w_inventory_.datasource_num_id\n",
        "                                )\n",
        "                                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh8_sq_w_inventory_\n",
        "                                    ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat8 = w_prod_cat_dh8_sq_w_inventory_.integration_id\n",
        "                                   AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh8_sq_w_inventory_.datasource_num_id\n",
        "                            )\n",
        "                            LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh9_sq_w_inventory_\n",
        "                                ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat9 = w_prod_cat_dh9_sq_w_inventory_.integration_id\n",
        "                               AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh9_sq_w_inventory_.datasource_num_id\n",
        "                        )\n",
        "                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh10_sq_w_inventory\n",
        "                            ON sq_w_inventory_product_ds_sq_w_in.inv_prod_cat10 = w_prod_cat_dh10_sq_w_inventory.integration_id\n",
        "                           AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh10_sq_w_inventory.datasource_num_id\n",
        "                    )\n",
        "                    LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS w_prod_cat_dh_unspsc_sq_w_inve\n",
        "                        ON sq_w_inventory_product_ds_sq_w_in.unspsc_code = w_prod_cat_dh_unspsc_sq_w_inve.integration_id\n",
        "                       AND sq_w_inventory_product_ds_sq_w_in.datasource_num_id = w_prod_cat_dh_unspsc_sq_w_inve.datasource_num_id\n",
        "                WHERE\n",
        "                    (1 = 1)\n",
        "            ) AS sq_w_inventory_product_ds_sq_w\n",
        "            LEFT OUTER JOIN\n",
        "            (\n",
        "                SELECT\n",
        "                    w_busn_location_d_lkp_w_busn_l.datasource_num_id AS datasource_num_id,\n",
        "                    w_busn_location_d_lkp_w_busn_l.row_wid AS row_wid,\n",
        "                    w_busn_location_d_lkp_w_busn_l.integration_id AS integration_id,\n",
        "                    w_busn_location_d_lkp_w_busn_l.effective_to_dt AS effective_to_dt,\n",
        "                    w_busn_location_d_lkp_w_busn_l.effective_from_dt AS effective_from_dt\n",
        "                FROM\n",
        "                    workspace.prxbi_dw.w_busn_location_d AS w_busn_location_d_lkp_w_busn_l\n",
        "                WHERE\n",
        "                    (1 = 1)\n",
        "            ) AS lkp_w_busn_location_d_lkp_w_bu\n",
        "                ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_busn_location_d_lkp_w_bu.datasource_num_id\n",
        "               AND sq_w_inventory_product_ds_sq_w.plant_loc_id = lkp_w_busn_location_d_lkp_w_bu.integration_id\n",
        "               AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_busn_location_d_lkp_w_bu.effective_from_dt\n",
        "               AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_busn_location_d_lkp_w_bu.effective_to_dt\n",
        "    )\n",
        "    LEFT OUTER JOIN workspace.prxbi_dw.w_int_org_d AS lkp_w_int_org_d_inventory\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_int_org_d_inventory.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.inventory_org_id = lkp_w_int_org_d_inventory.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_int_org_d_inventory.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_int_org_d_inventory.effective_to_dt\n",
        "    LEFT OUTER JOIN workspace.prxbi_dw.w_product_d AS lkp_w_product_d_product_wid\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_product_d_product_wid.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.product_id = lkp_w_product_d_product_wid.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt >= lkp_w_product_d_product_wid.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.created_on_dt < lkp_w_product_d_product_wid.effective_to_dt\n",
        "    LEFT OUTER JOIN\n",
        "    (\n",
        "        SELECT\n",
        "            w_user_d_lkp_w_user_d_changed_.datasource_num_id AS datasource_num_id,\n",
        "            w_user_d_lkp_w_user_d_changed_.row_wid AS row_wid,\n",
        "            w_user_d_lkp_w_user_d_changed_.integration_id AS integration_id,\n",
        "            w_user_d_lkp_w_user_d_changed_.effective_to_dt AS effective_to_dt,\n",
        "            w_user_d_lkp_w_user_d_changed_.effective_from_dt AS effective_from_dt\n",
        "        FROM\n",
        "            workspace.prxbi_dw.w_user_d AS w_user_d_lkp_w_user_d_changed_\n",
        "        WHERE\n",
        "                (1 = 1)\n",
        "            AND (w_user_d_lkp_w_user_d_changed_.delete_flg = 'N')\n",
        "    ) AS lkp_w_user_d_lkp_w_user_d_chan\n",
        "        ON sq_w_inventory_product_ds_sq_w.datasource_num_id = lkp_w_user_d_lkp_w_user_d_chan.datasource_num_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_by_id = lkp_w_user_d_lkp_w_user_d_chan.integration_id\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_on_dt >= lkp_w_user_d_lkp_w_user_d_chan.effective_from_dt\n",
        "       AND sq_w_inventory_product_ds_sq_w.changed_on_dt < lkp_w_user_d_lkp_w_user_d_chan.effective_to_dt\n",
        "    LEFT OUTER JOIN\n",
        "    (\n",
        "        SELECT\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.datasource_num_id AS datasource_num_id,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.row_wid AS row_wid,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.integration_id AS integration_id,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.effective_to_dt AS effective_to_dt,\n",
        "            lkp_w_user_d_lkp_w_user_d_crea.effective_from_dt AS effective_from_dt\n",
        "        FROM\n",
        "            (\n",
        "                SELECT\n",
        "                    w_user_d_lkp_w_user_d_created_.datasource_num_id AS datasource_num_id,\n",
        "                    w_user_d_lkp_w_user_d_created_.row_wid AS row_wid,\n",
        "                    w_user_d_lkp_w_user_d_created_.integration_id AS integration_id,\n",
        "                    w_user_d_lkp_w_user_d_created_.effective_to_dt AS effective_to_dt,\n",
        "                    w_user_d_lkp_w_user_d_created_.effective_from_dt AS effective_from_dt\n",
        "                FROM\n",
        "                    workspace.prxbi_dw.w_user_d AS w_user_d_lkp_w_user_d_created_\n",
        "                WHERE\n",
        "                        (1 = 1)\n",
        "                    AND (w_user_d_lkp_w_user_d_created_.delete_flg = 'N')\n",
        "            ) AS lkp_w_user_d_lkp_w_user_d_crea\n",
        "    ) AS lkp_w_user_d_lkp_w_user_d_cr_1\n",
        "        ON inline_view.datasource_num_id = lkp_w_user_d_lkp_w_user_d_cr_1.datasource_num_id\n",
        "       AND inline_view.created_by_id = lkp_w_user_d_lkp_w_user_d_cr_1.integration_id\n",
        "       AND inline_view.created_on_dt >= lkp_w_user_d_lkp_w_user_d_cr_1.effective_from_dt\n",
        "       AND inline_view.created_on_dt < lkp_w_user_d_lkp_w_user_d_cr_1.effective_to_dt\n",
        "    WHERE (1 = 1)\n",
        ") AS C\n",
        "LEFT OUTER JOIN workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "    ON C.src_eff_from_dt = T.src_eff_from_dt\n",
        "   AND C.datasource_num_id = T.datasource_num_id\n",
        "   AND C.integration_id = T.integration_id;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.i_inventory_product_d_flow;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table Optimization (SCEN_TASK_NO {150})\n",
        "\n",
        "Oracle index creation is not directly applicable to Delta Lake. For performance, ZORDER may be used if this table were persistent and heavily queried. As a temporary flow table, explicit ZORDER is usually not necessary but is included as a best practice example if it were a persistent table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
        "OPTIMIZE workspace.prxbi_dw.i_inventory_product_d_flow ZORDER BY (src_eff_from_dt, datasource_num_id, integration_id);"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error Logging & Other Bypassed Steps (SCEN_TASK_NO {140} - {320})\n",
        "\n",
        "Many ODI tasks related to error logging and specific detection strategies are either bypassed, not applicable, or handled differently in Databricks. These steps have been either removed (PL/SQL blocks, `ALTER SESSION`) or noted as comments where the original ODI task was bypassed or its functionality is inherently different in Delta Lake."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Merge into Target Table (SCEN_TASK_NO {330} - {350})"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {330} & {350}: Combined Oracle UPDATE and INSERT into a single MERGE statement\n",
        "MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "USING workspace.prxbi_dw.i_inventory_product_d_flow AS S\n",
        "ON\n",
        "        T.src_eff_from_dt = S.src_eff_from_dt\n",
        "    AND T.datasource_num_id = S.datasource_num_id\n",
        "    AND T.integration_id = S.integration_id\n",
        "WHEN MATCHED AND S.ind_update = 'U' THEN UPDATE SET\n",
        "    T.product_wid = S.product_wid,\n",
        "    T.inventory_org_wid = S.inventory_org_wid,\n",
        "    T.plant_loc_wid = S.plant_loc_wid,\n",
        "    T.product_num = S.product_num,\n",
        "    T.abc_ind = S.abc_ind,\n",
        "    T.planner_code = S.planner_code,\n",
        "    T.procurement_type_code = S.procurement_type_code,\n",
        "    T.spc_proc_type_code = S.spc_proc_type_code,\n",
        "    T.buyer_code = S.buyer_code,\n",
        "    T.buyer_name = S.buyer_name,\n",
        "    T.commodity_code = S.commodity_code,\n",
        "    T.commodity_uom_code = S.commodity_uom_code,\n",
        "    T.profit_center_num = S.profit_center_num,\n",
        "    T.reorder_point = S.reorder_point,\n",
        "    T.safety_stock_level = S.safety_stock_level,\n",
        "    T.min_lot_size = S.min_lot_size,\n",
        "    T.max_lot_size = S.max_lot_size,\n",
        "    T.fixed_lot_size = S.fixed_lot_size,\n",
        "    T.max_stock_level = S.max_stock_level,\n",
        "    T.lot_ordering_cost = S.lot_ordering_cost,\n",
        "    T.mrp_time_fence = S.mrp_time_fence,\n",
        "    T.ext_procure_time = S.ext_procure_time,\n",
        "    T.internal_mfg_time = S.internal_mfg_time,\n",
        "    T.max_storage_days = S.max_storage_days,\n",
        "    T.mrp_profile_code = S.mrp_profile_code,\n",
        "    T.mrp_type_code = S.mrp_type_code,\n",
        "    T.mrp_grp_code = S.mrp_grp_code,\n",
        "    T.lot_size_code = S.lot_size_code,\n",
        "    T.backflush_ind = S.backflush_ind,\n",
        "    T.qa_inspect_ind = S.qa_inspect_ind,\n",
        "    T.repetitive_mfg_ind = S.repetitive_mfg_ind,\n",
        "    T.bulk_item_ind = S.bulk_item_ind,\n",
        "    T.forecast_period = S.forecast_period,\n",
        "    T.mfg_uom_code = S.mfg_uom_code,\n",
        "    T.issue_uom_code = S.issue_uom_code,\n",
        "    T.manufacturing_place = S.manufacturing_place,\n",
        "    T.loading_type_code = S.loading_type_code,\n",
        "    T.int_store_loc_code = S.int_store_loc_code,\n",
        "    T.ext_store_loc_code = S.ext_store_loc_code,\n",
        "    T.active_flg = S.active_flg,\n",
        "    T.created_by_wid = S.created_by_wid,\n",
        "    T.changed_by_wid = S.changed_by_wid,\n",
        "    T.created_on_dt = S.created_on_dt,\n",
        "    T.changed_on_dt = S.changed_on_dt,\n",
        "    T.aux1_changed_on_dt = S.aux1_changed_on_dt,\n",
        "    T.aux2_changed_on_dt = S.aux2_changed_on_dt,\n",
        "    T.aux3_changed_on_dt = S.aux3_changed_on_dt,\n",
        "    T.aux4_changed_on_dt = S.aux4_changed_on_dt,\n",
        "    T.src_eff_to_dt = S.src_eff_to_dt,\n",
        "    T.effective_from_dt = S.effective_from_dt,\n",
        "    T.delete_flg = S.delete_flg,\n",
        "    T.tenant_id = S.tenant_id,\n",
        "    T.x_custom = S.x_custom,\n",
        "    T.inv_prod_cat1 = S.inv_prod_cat1,\n",
        "    T.inv_prod_cat2 = S.inv_prod_cat2,\n",
        "    T.inv_prod_cat3 = S.inv_prod_cat3,\n",
        "    T.inv_prod_cat4 = S.inv_prod_cat4,\n",
        "    T.inv_prod_cat5 = S.inv_prod_cat5,\n",
        "    T.inv_prod_cat6 = S.inv_prod_cat6,\n",
        "    T.inv_prod_cat7 = S.inv_prod_cat7,\n",
        "    T.inv_prod_cat8 = S.inv_prod_cat8,\n",
        "    T.inv_prod_cat9 = S.inv_prod_cat9,\n",
        "    T.inv_prod_cat10 = S.inv_prod_cat10,\n",
        "    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid,\n",
        "    T.inv_prod_cat2_wid = S.inv_prod_cat2_wid,\n",
        "    T.inv_prod_cat3_wid = S.inv_prod_cat3_wid,\n",
        "    T.inv_prod_cat4_wid = S.inv_prod_cat4_wid,\n",
        "    T.inv_prod_cat5_wid = S.inv_prod_cat5_wid,\n",
        "    T.inv_prod_cat6_wid = S.inv_prod_cat6_wid,\n",
        "    T.inv_prod_cat7_wid = S.inv_prod_cat7_wid,\n",
        "    T.inv_prod_cat8_wid = S.inv_prod_cat8_wid,\n",
        "    T.inv_prod_cat9_wid = S.inv_prod_cat9_wid,\n",
        "    T.inv_prod_cat10_wid = S.inv_prod_cat10_wid,\n",
        "    T.invoiceable_item_flag = S.invoiceable_item_flag,\n",
        "    T.invoice_enabled_flag = S.invoice_enabled_flag,\n",
        "    T.primary_uom_code = S.primary_uom_code,\n",
        "    T.c_primary_uom_code = S.c_primary_uom_code,\n",
        "    T.unspsc_code = S.unspsc_code,\n",
        "    T.unspsc_inv_prod_cat_wid = S.unspsc_inv_prod_cat_wid,\n",
        "    T.commodity_name = S.commodity_name,\n",
        "    T.commodity_uom_name = S.commodity_uom_name,\n",
        "    T.ext_store_loc_name = S.ext_store_loc_name,\n",
        "    T.int_store_loc_name = S.int_store_loc_name,\n",
        "    T.issue_uom_name = S.issue_uom_name,\n",
        "    T.loading_type_name = S.loading_type_name,\n",
        "    T.lot_size_name = S.lot_size_name,\n",
        "    T.mfg_uom_name = S.mfg_uom_name,\n",
        "    T.mrp_grp_name = S.mrp_grp_name,\n",
        "    T.mrp_profile_name = S.mrp_profile_name,\n",
        "    T.mrp_type_name = S.mrp_type_name,\n",
        "    T.planner_name = S.planner_name,\n",
        "    T.primary_uom_name = S.primary_uom_name,\n",
        "    T.procurement_type_name = S.procurement_type_name,\n",
        "    T.profit_center_name = S.profit_center_name,\n",
        "    T.spc_proc_type_name = S.spc_proc_type_name,\n",
        "    T.status_code = S.status_code,\n",
        "    T.w_status_code = S.w_status_code,\n",
        "    T.product_type_code = S.product_type_code,\n",
        "    T.make_buy_ind = S.make_buy_ind,\n",
        "    T.fixed_lead_time = S.fixed_lead_time,\n",
        "    T.variable_lead_time = S.variable_lead_time,\n",
        "    T.cumulative_total_lead_time = S.cumulative_total_lead_time,\n",
        "    T.postprocessing_lead_time = S.postprocessing_lead_time,\n",
        "    T.preprocessing_lead_time = S.preprocessing_lead_time,\n",
        "    T.process_quality_enabled_flg = S.process_quality_enabled_flg,\n",
        "    T.x_price_sequence = S.x_price_sequence,\n",
        "    T.x_organization_name = S.x_organization_name,\n",
        "    T.x_product_desc = S.x_product_desc,\n",
        "    T.x_uom_desc = S.x_uom_desc,\n",
        "    T.x_inv_item_flg = S.x_inv_item_flg,\n",
        "    T.x_stock_item_flg = S.x_stock_item_flg,\n",
        "    T.x_trans_flg = S.x_trans_flg,\n",
        "    T.x_rev_flg = S.x_rev_flg,\n",
        "    T.x_cost_flg = S.x_cost_flg,\n",
        "    T.x_gcoa_acct = S.x_gcoa_acct,\n",
        "    T.x_gcoa_prod = S.x_gcoa_prod,\n",
        "    T.x_tax_cat = S.x_tax_cat,\n",
        "    T.organization_id = S.organization_id,\n",
        "    T.x_gcoa_loc_acct = S.x_gcoa_loc_acct,\n",
        "    T.effective_to_dt = to_timestamp('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss'),\n",
        "    T.current_flg = 'Y',\n",
        "    T.w_update_dt = current_timestamp(),\n",
        "    T.etl_proc_wid = ${ETL_PROC_WID}\n",
        "WHEN NOT MATCHED AND S.ind_update = 'I' THEN INSERT (\n",
        "    product_wid,\n",
        "    inventory_org_wid,\n",
        "    plant_loc_wid,\n",
        "    product_num,\n",
        "    abc_ind,\n",
        "    planner_code,\n",
        "    procurement_type_code,\n",
        "    spc_proc_type_code,\n",
        "    buyer_code,\n",
        "    buyer_name,\n",
        "    commodity_code,\n",
        "    commodity_uom_code,\n",
        "    profit_center_num,\n",
        "    reorder_point,\n",
        "    safety_stock_level,\n",
        "    min_lot_size,\n",
        "    max_lot_size,\n",
        "    fixed_lot_size,\n",
        "    max_stock_level,\n",
        "    lot_ordering_cost,\n",
        "    mrp_time_fence,\n",
        "    ext_procure_time,\n",
        "    internal_mfg_time,\n",
        "    max_storage_days,\n",
        "    mrp_profile_code,\n",
        "    mrp_type_code,\n",
        "    mrp_grp_code,\n",
        "    lot_size_code,\n",
        "    backflush_ind,\n",
        "    qa_inspect_ind,\n",
        "    repetitive_mfg_ind,\n",
        "    bulk_item_ind,\n",
        "    forecast_period,\n",
        "    mfg_uom_code,\n",
        "    issue_uom_code,\n",
        "    manufacturing_place,\n",
        "    loading_type_code,\n",
        "    int_store_loc_code,\n",
        "    ext_store_loc_code,\n",
        "    active_flg,\n",
        "    created_by_wid,\n",
        "    changed_by_wid,\n",
        "    created_on_dt,\n",
        "    changed_on_dt,\n",
        "    aux1_changed_on_dt,\n",
        "    aux2_changed_on_dt,\n",
        "    aux3_changed_on_dt,\n",
        "    aux4_changed_on_dt,\n",
        "    src_eff_from_dt,\n",
        "    src_eff_to_dt,\n",
        "    effective_from_dt,\n",
        "    delete_flg,\n",
        "    datasource_num_id,\n",
        "    integration_id,\n",
        "    tenant_id,\n",
        "    x_custom,\n",
        "    inv_prod_cat1,\n",
        "    inv_prod_cat2,\n",
        "    inv_prod_cat3,\n",
        "    inv_prod_cat4,\n",
        "    inv_prod_cat5,\n",
        "    inv_prod_cat6,\n",
        "    inv_prod_cat7,\n",
        "    inv_prod_cat8,\n",
        "    inv_prod_cat9,\n",
        "    inv_prod_cat10,\n",
        "    inv_prod_cat1_wid,\n",
        "    inv_prod_cat2_wid,\n",
        "    inv_prod_cat3_wid,\n",
        "    inv_prod_cat4_wid,\n",
        "    inv_prod_cat5_wid,\n",
        "    inv_prod_cat6_wid,\n",
        "    inv_prod_cat7_wid,\n",
        "    inv_prod_cat8_wid,\n",
        "    inv_prod_cat9_wid,\n",
        "    inv_prod_cat10_wid,\n",
        "    invoiceable_item_flag,\n",
        "    invoice_enabled_flag,\n",
        "    primary_uom_code,\n",
        "    c_primary_uom_code,\n",
        "    unspsc_code,\n",
        "    unspsc_inv_prod_cat_wid,\n",
        "    commodity_name,\n",
        "    commodity_uom_name,\n",
        "    ext_store_loc_name,\n",
        "    int_store_loc_name,\n",
        "    issue_uom_name,\n",
        "    loading_type_name,\n",
        "    lot_size_name,\n",
        "    mfg_uom_name,\n",
        "    mrp_grp_name,\n",
        "    mrp_profile_name,\n",
        "    mrp_type_name,\n",
        "    planner_name,\n",
        "    primary_uom_name,\n",
        "    procurement_type_name,\n",
        "    profit_center_name,\n",
        "    spc_proc_type_name,\n",
        "    status_code,\n",
        "    w_status_code,\n",
        "    product_type_code,\n",
        "    make_buy_ind,\n",
        "    fixed_lead_time,\n",
        "    variable_lead_time,\n",
        "    cumulative_total_lead_time,\n",